# RSNA Knee Abnormality Detection — CPU Tsetlin Machine, v4

**Lineage:** v2 scored **0.55** on the leaderboard; v3 added two feature maps on top of it.
This notebook keeps the composite-Tsetlin-Machine approach and rebuilds the four things that
were actually capping the score. The reference point is the 0.91 DINOv3 ensemble notebook in
this repo (`rsna-knee-v41-dinov3-e10-alpha060.ipynb`); everything below that could be lifted
from it without a GPU has been.

## What was capping v2 at 0.55

| # | Diagnosis | Evidence | Fix in v4 |
|---|---|---|---|
| 1 | **The training labels were the weak link, not the model.** Only **58** of ~4 400 training studies carry expert annotations. Everything else is supervised by the report miner — and v2's miner is a short English-first keyword list with a 28-character negation window. The corpus is multilingual (ES/FR/NL/DE/TR/HR/EL/BG). | v41 prints `58 studies carry the twelve annotations`; its own miner is scored against those 58 and against a corpus-wide silence rate | Port v41's report lexicon **verbatim** (§2): directional negation, normality/uncertainty polarity, severity and ICRS/Outerbridge grading, OA site attribution, ~9 languages. Scores are graded, not binarised, and come with a per-finding confidence used as a **sample weight**. |
| 2 | **v2's Tsetlin Machine could not converge in the time it was given.** Its `_update` is a Python loop over samples × clauses. | Measured on an easy held-out task (even-vs-odd digits, 256 thermometer features): v2's TM reaches **AUC 0.535 after 10 epochs (40 s)** and 0.919 after 25. At *identical* hyper-parameters the v4 TM reaches **0.940 in 1.3 s**, and **0.995** with clause weights on. | Rewrite the TM so clause evaluation and feedback accumulation are **float32 matrix products** (§3). ~30× faster at the sizes used here; that budget is what pays for slices, folds and resolution. |
| 3 | **Nothing measured whether the image model helped.** TM confidences went straight into the blend with no held-out estimate, so a TM at chance and a TM that learned looked the same. | v2 §7.4 has no validation of any kind | Grouped 4-fold **out-of-fold AUC for every member** (§6). Members are rank-mean combined with weight ∝ (OOF AUC − 0.5), and a finding whose best member cannot clear `GATE_AUC` falls back to the tabular model. **v4 cannot score below the tabular baseline on any finding.** |
| 4 | **One 32 px middle slice, picked by plane alone.** Left and right knees were fed in mirrored, field of view varied 120–200 mm so anatomy sat at different pixel scales, and effusion/synovitis were scored off whatever sequence happened to be sagittal. | v2 `read_middle_slice` / `TM_PLANE_OF_LABEL` | §5 ports v41's pixel pipeline: **slot** selection (plane × fluid-sensitive × fat-suppressed) with a per-finding slot prior, geometry-ordered slices, **several slices per slot** pooled per study (max / top-2 / mean, chosen per finding by OOF), **laterality normalisation**, and a **physical-size crop** (`CROP_MM`) so 1 pixel is the same number of millimetres in every study. |

## What this can and cannot reach

Be clear about the ceiling: 0.91 comes from ViT backbones self-supervised on millions of images and
fine-tuned on this competition, ensembled 25 ways. A Tsetlin Machine over 32–48 px binarised slices
is not going to match that. What the four fixes above are worth is the difference between *supervising
a model on noise* and *supervising it on labels that agree with the annotations* — plus, for the first
time, a number telling you which findings the image model has learned.

Expect the large, high-contrast, whole-joint findings (**Effusion, Baker's, the three OA compartments,
Synovitis**) to move well, and the small focal ones (**ACL, meniscal tears, Fracture**) to stay near
chance at this resolution — they are a few millimetres of altered signal on a few slices. The OOF
table in §7 tells you which is which on your run, and the gate means the ones that fail cost nothing.

## 1. Setup, data, and the knobs

Same minimal-download contract as v2: on Kaggle everything is mounted, locally only the five small
CSVs are fetched (never the ~570 GB of DICOMs). The image sections skip themselves when
`train_series/` is not present, so the notebook runs end to end either way.

In [ ]:
import os, re, gc, json, time, math, subprocess, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

T0 = time.time()
def log(msg):
    print(f"[{time.time()-T0:7.1f}s] {msg}", flush=True)

COMPETITION = "rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
          "Medial OA", "Lateral OA", "PF OA", "Effusion",
          "Synovitis", "Baker's", "Contusion", "Fracture"]

def find_root():
    for c in [Path("/kaggle/input/competitions/" + COMPETITION),
              Path("/kaggle/input/" + COMPETITION), Path("data"), Path(".")]:
        if (c / "train.csv").is_file():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for d1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [d1] + sorted(p for p in d1.iterdir() if p.is_dir()):
                if (cand / "train.csv").is_file():
                    return cand
    return None

ON_KAGGLE = Path("/kaggle/input").is_dir()
ROOT = find_root()
if ROOT is None:
    ROOT = Path("data"); ROOT.mkdir(exist_ok=True)
HAVE_IMAGES = (ROOT / "train_series").is_dir() and (ROOT / "test_series").is_dir()

# ---- budget -------------------------------------------------------------
# Kaggle allows 12 h on CPU. Stop enrolling ensemble members at TIME_BUDGET and
# ship what has been banked; submission.csv is rewritten after every member, so
# a hard stop always leaves a valid, best-so-far file on disk.
TIME_BUDGET = float(os.environ.get("RSNA_TIME_BUDGET", 8 * 3600))
def time_left():
    return TIME_BUDGET - (time.time() - T0)

STAGE_SECS = {}          # stage name -> seconds, reported in feedback.txt

# ---- pixels -------------------------------------------------------------
CACHE_IMG   = 64        # slices are cached at this size ...
TM_IMG      = 32        # ... and fed to the TM at this one (raise with the clause budget)
CROP_MM     = 160.0     # physical field of view kept around the joint centre
SLICE_BAND  = (0.25, 0.75)   # fraction of the ordered stack the slices are drawn from
N_SLICE     = 3         # slices per slot per study
IO_THREADS  = 8

# ---- Tsetlin Machine ----------------------------------------------------
TM_CLAUSES      = 250
TM_T            = 1200
TM_S            = 5.0
TM_EPOCHS       = 6
TM_BATCH        = 64
TM_MAX_INCLUDED = 32
TM_WEIGHTED     = True     # clause weights: worth ~0.05 AUC on the digits check below
TM_METHODS      = ["thermo", "sobel", "adaptive_mean"]   # also available: otsu, hog, color
TM_THERMO_BITS  = 4
TM_HOG_BINS     = 4
TM_COLOR_WINDOWS = 3

# ---- supervision & validation ------------------------------------------
N_TRAIN     = 2500     # studies whose pixels are read for training
MIN_TRAIN   = 200      # below this many usable studies the image model is not worth fitting
N_FOLDS     = 4
SEED        = 42
POS_FLOOR   = 0.25     # oversample positives in the training half up to this rate
GATE_AUC    = 0.53     # a finding keeps its image model only above this OOF AUC
SLOTS_PER_TARGET = 2   # how many slots per finding to try, in prior order
INCLUDE_PROBE = True   # also fit a linear probe on the same bits, as a yardstick

print("on kaggle:", ON_KAGGLE, "| root:", ROOT, "| images mounted:", HAVE_IMAGES)
print("time budget: %.1f h" % (TIME_BUDGET / 3600))

In [ ]:
MINIMAL_FILES = ["train.csv", "train_series.csv", "test.csv",
                 "test_series.csv", "sample_submission.csv"]

def ensure_minimal_data():
    missing = [f for f in MINIMAL_FILES if not (ROOT / f).is_file()]
    if not missing:
        return
    if ON_KAGGLE:
        raise SystemExit("Competition data not mounted. Add '%s' as an input." % COMPETITION)
    print("downloading (small) missing files:", missing)
    for f in missing:
        subprocess.run(["kaggle", "competitions", "download", "-c", COMPETITION,
                        "-f", f, "-p", str(ROOT)], check=True, capture_output=True, text=True)

ensure_minimal_data()

def load(name):
    return pd.read_csv(ROOT / name, dtype={"StudyInstanceUID": str})

train        = load("train.csv")
train_series = load("train_series.csv")
test         = load("test.csv")
test_series  = load("test_series.csv")
sample_sub   = load("sample_submission.csv")

TEST_UID = test["StudyInstanceUID"].tolist()
PREVALENCE = train[LABELS].mean()

# A valid submission exists from this point on; every later stage overwrites it.
_bench = pd.DataFrame({"StudyInstanceUID": TEST_UID})
for c in LABELS:
    _bench[c] = float(PREVALENCE[c])
_bench[sample_sub.columns].to_csv("submission.csv", index=False)

log("train %s  test %s  train_series %s" % (train.shape, test.shape, train_series.shape))
log("studies carrying all twelve expert annotations: %d of %d"
    % (int(train[LABELS].notna().all(axis=1).sum()), len(train)))

## 2. Report labels — the lexicon from the 0.91 notebook, verbatim

With 58 expert-annotated studies out of ~4 400, **the report miner is the training set**. A better
miner is therefore worth more than any change to the image model, and there is no reason to write
one: the next three cells are copied unchanged from `rsna-knee-v41-dinov3-e10-alpha060.ipynb`
(credit there: pilkwang, stevenleehans, lixin73).

Over v2's miner it adds: negation that respects direction and distance from the finding
(`_negated`, 90-char window, cancelled by *but/however/pero/ancak/…*), a separate normality channel
(*intact, preserved, uredan, φυσιολογικ, o.B.*), an uncertainty channel that scores as a soft
positive, severity and ICRS/Outerbridge grades folded into the score, OA evidence attributed to the
medial / lateral / patellofemoral site by proximity, a global-gonarthrosis inheritance rule, decoys
(*microfracture*, *parameniscal cyst*), and a synovitis back-off that borrows from effusion when the
report never mentions the synovium. It returns a graded score in `[0, 1]` plus `__conf`, `__npos`,
`__nneg` per finding.

In [ ]:
from __future__ import annotations
import re
import unicodedata
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_PRE = str.maketrans({'ı': 'i', 'İ': 'i', 'I': 'i', 'ß': 'ss', 'đ': 'd', 'Đ': 'd', 'ø': 'o', 'Ø': 'o', 'æ': 'ae', 'Æ': 'ae'})

def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join((ch for ch in text if not unicodedata.combining(ch)))
    text = text.replace('\xad', '')
    text = re.sub('[_\\-/\\\\]+', ' ', text)
    text = re.sub('[ \\t]+', ' ', text)
    return text
_SENT_SPLIT = re.compile('(?<=[.;!?])\\s+|\\n+')

def unwrap(text: str) -> str:
    if not isinstance(text, str):
        return ''
    out = []
    for line in text.split('\n'):
        s = line.strip()
        if out and out[-1] and (not re.search('[.;:!?>*•]$', out[-1])) and (len(out[-1].split()) >= 4) and s and (not s[:1].isupper()):
            out[-1] = out[-1] + ' ' + s
        else:
            out.append(s)
    return '\n'.join(out)

def clauses(text: str):
    norm = normalize(unwrap(text) if FEATURES['unwrap'] else text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]
    merged = []
    for i, c in enumerate(raw):
        if c.endswith(':') and len(c.split()) <= 14 and (i + 1 < len(raw)):
            merged.append(c + ' ' + raw[i + 1])
        merged.append(c)
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend((p.strip() for p in c.split(',') if len(p.split()) > 2))
    return out
FEATURES = {'unwrap': True, 'directional_negation': True, 'oa_inherit': True, 'graded_pathology': True, 'synovitis_backoff': True}

def _rx(*alts: str) -> re.Pattern:
    return re.compile('|'.join(alts))
PRE_NEG = _rx('\\bno\\b', '\\bnot\\b', '\\bwithout\\b', '\\bnegative for\\b', '\\babsence\\b', '\\bno evidence\\b', '\\bfree of\\b', '\\bnone\\b', '\\bneither\\b', '\\bnor\\b', '\\bsin\\b', '\\bno hay\\b', '\\bausencia\\b', '\\bausentes?\\b', '\\bno se\\b', '\\bpas de\\b', '\\bsans\\b', '\\baucune?\\b', '\\bgeen\\b', '\\bzonder\\b', '\\bniet\\b', '\\bkeine?[nmrs]?\\b', '\\bohne\\b', '\\bnicht\\b', '\\bkein\\b', '\\bnema\\b', '\\bbez\\b', '\\bnisu\\b', '\\bnije\\b', '\\bδεν\\b', '\\bχωρις\\b', 'ουδεν', '\\bουτε\\b', '\\bбез\\b', '\\bне\\b', 'липсва', '\\bняма\\b')
POST_NEG = _rx('\\byok\\b', '\\byoktur\\b', 'izlenmemekte', 'saptanmadi', '\\bdegil\\b', 'gozlenmemekte', 'mevcut degil', 'eslik etmiyor', '\\bizlenmedi\\b', 'izlenmemistir', 'saptanmamistir', 'gorulmemistir', '\\bnema znakova\\b', 'bez znakova')
NEGATION = _rx(PRE_NEG.pattern, POST_NEG.pattern, '\\bunremarkable\\b')
NEG_WINDOW = 90

def _negated(clause: str, start: int, end: int) -> bool:
    for m in PRE_NEG.finditer(clause):
        if m.end() <= start and start - m.end() <= NEG_WINDOW:
            if not re.search('\\b(but|however|ancak|fakat|pero|maar|aber|no i|ali|ωστοσο|αλλα|но)\\b', clause[m.end():start]):
                return True
    for m in POST_NEG.finditer(clause):
        if m.start() >= end and m.start() - end <= NEG_WINDOW:
            return True
    return False
NORMALITY = _rx('\\bnormal', '\\bintact\\b', '\\bpreserved\\b', '\\bwithin normal limits\\b', 'limites normales', '\\bconservad', '\\bintegr', '\\bnormales\\b', '\\bdoga(l|ll)\\b', 'korunmus', '\\bnormaldir\\b', 'olagan', '\\buredn', '\\bocuvan', '\\bodrzan', '\\bintakt', '\\bprimjeren', '\\bodrzanog kontinuiteta', '\\bodržan', 'φυσιολογικ', 'ακεραι', 'δεν παρατηρουνται', 'δεν σημειωνονται', 'unauffallig', 'regelrecht', '\\bo\\.?b\\.?\\b', 'нормал', 'запазен', 'съхранен', '\\bбез особености\\b', 'интактн', '\\bgaaf\\b', '\\bnormaal\\b')
NORMAL_PHRASE = _rx('\\bsin alteracion', '\\bsin cambios\\b', '\\bsin particularidad', '\\bsin hallazgos\\b', '\\bsin lesion', '\\bsin signos de (rotura|lesion)', '\\bcontinu[oa]s?\\b', '\\bcontinuidad conservada\\b', '\\bno abnormalit', '\\bno significant abnormalit', '\\bunremarkable\\b', '\\bno evidence of (tear|injury|abnormalit)', '\\bohne auffalligkeit', '\\bkein nachweis\\b', '\\bohne befund\\b', '\\bgeen afwijking', '\\bzonder afwijking', '\\bsans anomalie', "\\bpas d[e']anomalie", '\\bbez osobitosti\\b', '\\bbez znakova (rupture|lezije)\\b', '\\bbez patoloskih\\b', 'χωρις αλλοιωσ', 'χωρις παθολογ', 'δεν παρατηρουνται (αξιολογα|παθολογ)', '\\bбез особености\\b', '\\bбез патологич', '\\bбез данни за\\b', '\\bozel bir ozellik yok', '\\bpatolojik bulgu (yok|izlenmemis)')
UNCERTAIN = _rx('\\bpossible\\b', '\\bprobable\\b', '\\bsuspicious\\b', '\\bsuspected?\\b', 'cannot (be )?exclude', '\\bmay\\b', '\\bquestionable\\b', '\\bequivocal\\b', '\\br/o\\b', '\\bdd\\b', '\\blikely\\b', '\\bsuggest', '\\bcompatible with\\b', '\\bposible\\b', 'sin criterios categoricos', '\\bdudos', '\\bsugier', '\\bmuhtemel\\b', '\\bolasi\\b', '\\bsupheli\\b', '\\bizlenim', '\\bdusundur', '\\bmoguce\\b', '\\bvjerojatno\\b', '\\bsumnja\\b', '\\bmoze odgovarati\\b', 'πιθαν', 'υποπτ', '\\bmoglich', '\\bverdachtig', '\\bfraglich', '\\bv\\.?a\\.?\\b', '\\bwohl\\b', '\\bвъзможно\\b', '\\bвероятно\\b', 'суспект', '\\bmogelijk\\b', '\\bverdacht\\b')

In [ ]:
TEAR = _rx('\\btear', '\\btorn\\b', '\\brupture', '\\bdisruption\\b', 'discontinuit', '\\bavuls', '\\bmacerat', '\\bbuckethandle\\b', 'bucket handle', '\\brotura\\b', '\\broturas\\b', '\\bruptura', '\\bdesgarro', '\\broto\\b', '\\bdechirure', '\\bdechire', '\\bscheur', '\\bruptuur', 'gescheurd', '\\briss\\b', 'einriss', '\\bruptur', 'zerreiss', '\\blasion', '\\bausriss', '\\byirtik', '\\byirtig', '\\bkopma\\b', 'butunluk kaybi', '\\brupturu\\b', 'devamsizlik', '\\brupture\\b', '\\bdevamliligi secilememis', '\\bpuknuce', '\\bprekid\\b', '\\bpukotin', '\\bruptur', 'ρηξη', 'ρηξις', 'ρηγμα', 'ασυνεχεια', 'руптура', 'разкъсв', 'разрив', 'скъсв', '\\bлезия\\b')
DEGEN = _rx('degenerat', '\\bmucoid\\b', '\\bmyxoid\\b', '\\bfray', '\\bfissur', 'dejeneratif', '\\bmukoid\\b', 'degenerativn', 'εκφυλ', 'дегенерат', '\\bμυξοειδ', '\\bμυξωδ', '\\bmeniskopat', '\\bmeniscopath', '\\bmuco ?ide\\b', 'aufgefasert', '\\bdejenerasyon\\b')
INJURY = _rx('\\binjur', '\\bsprain', '\\blesion', '\\blasion', '\\bedema\\b', '\\boedema\\b', '\\bodem\\b', '\\bedem\\b', '\\bοιδημα', '\\bодем', '\\bедем', '\\bstrain\\b', '\\bhigh signal\\b', '\\bsignal alteration\\b', '\\bhiperintens', '\\bhyperintens', 'aumento de senal', 'alteracion de senal', 'cambio de senal', '\\bsignalanhebung', '\\bsignalalteration', 'verhoogd signaal', 'sinyal artis', 'αυξημενο σημα', 'повишен сигнал', '\\besguince\\b', '\\bthicken', '\\bzadebljanje\\b', '\\bverdikking\\b', '\\bdistenzij', '\\blaksite\\b', '\\blaxity\\b', '\\bpartial\\b', '\\bparcijaln', '\\bparcial', '\\bpartiel', '\\bpartiell')
_GRADE_RX = re.compile('(?:grade|grad|grado|grau|derece|stupnja|stupanj|βαθμ|степен|icrs|outerbridge)[\\s:]*(?:grade\\s*)?([1-4]|iv|iii|ii|i)\\b')
_ROMAN = {'i': 1, 'ii': 2, 'iii': 3, 'iv': 4}

def _grade_of(clause: str):
    best = None
    for m in _GRADE_RX.finditer(clause):
        v = m.group(1)
        n = _ROMAN.get(v, None) if not v.isdigit() else int(v)
        if n is not None and (best is None or n > best):
            best = n
    return best
ANAT = {'ACL': _rx('anterior cruciate', '\\bacl\\b', 'cruzado anterior', '\\blca\\b', 'croise anterieur', 'voorste kruisband', '\\bvkb\\b', 'vorderes kreuzband', 'vorderen kreuzband', 'vordere kreuzband', 'on capraz', '\\bocb\\b', 'anterior capraz', 'prednji krizni', 'prednjeg krizn', 'προσθι[οα][^ ]* χιαστ', 'προσθιου χιαστου', 'χιαστο[^ ]* συνδεσμ', '\\bχιαστ\\w*', 'предна кръстна', 'предната кръстна', 'предна кръста', 'cruciate ligaments', 'ligamentos cruzados', 'ligaments croises', 'kruisbanden', 'kreuzbander', 'capraz baglar', 'krizn[a-z]* ligament[a-z]*', 'χιαστοι συνδεσμ', 'χιαστων συνδεσμ', 'кръстните връзки', 'кръстни връзки'), 'MCL': _rx('medial collateral', '\\bmcl\\b', 'tibial collateral', 'colateral medial', 'colateral interno', '\\blcm\\b', 'collateral medial', 'collateral interne', 'mediale collaterale', 'binnenband', '\\b(mediale|laterale) banden\\b', '\\bcollaterale banden\\b', 'innenband', 'mediales? kollateral', '\\bic yan bag', 'medial kollateral', '\\biyb\\b', 'medyal kollateral', 'medijalni kolateraln', 'medijalnog kolateraln', 'εσω πλαγι', 'εσωτερικο πλαγι', '\\bπλαγι\\w* συνδεσμ', '\\bπλαγιοι\\b', 'медиален колатерал', 'вътрешна странична', '\\bколатерал\\w*', '\\bcolaterales\\b', '\\bcollateraux\\b', '\\bcollateralen\\b', '\\bkolateralni\\b', 'collateral ligaments', 'ligamentos colaterales', 'ligaments collateraux', 'collaterale banden', 'kollateralbander', 'seitenbander', 'yan baglar', 'kolateraln[a-z]* ligament[a-z]*', 'πλαγιοι συνδεσμ', 'πλαγιων συνδεσμ', 'колатерални връзки', 'страничните връзки'), 'Medial Meniscus': _rx('medial meniscus', '\\bmm\\b(?= tear)', 'medial menisc', 'menisco medial', 'menisco interno', 'menisque medial', 'menisque interne', 'mediale meniscus', 'binnenmeniscus', 'innenmeniskus', 'medialen? meniskus', 'innenmeniskushinterhorn', 'medyal menisk', '\\bic menisk', 'medijalni meniskus', 'medijalnog meniskusa', 'medijalnom meniskusu', 'medijaln\\w* menisk\\w*', '\\bmedijalnog meniska\\b', 'medijalni menisk', 'εσω μηνισκ', 'μηνισκ[^ ]* του εσω', 'εσω διαμερισμα[^.]{0,40}μηνισκ', 'медиалния менискус', 'медиален менискус', 'вътрешния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'amfoteroi\\w* mhnisk', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc'), 'Lateral Meniscus': _rx('lateral meniscus', 'lateral menisc', 'menisco lateral', 'menisco externo', 'menisque lateral', 'menisque externe', 'laterale meniscus', 'buitenmeniscus', 'aussenmeniskus', 'lateralen? meniskus', 'aussenmeniskushinterhorn', 'lateral menisk', '\\bdis menisk', 'lateralni meniskus', 'lateralnog meniskusa', 'lateralnom meniskusu', 'lateraln\\w* menisk\\w*', '\\blateralnog meniska\\b', 'εξω μηνισκ', 'μηνισκ[^ ]* του εξω', 'εξω διαμερισμα[^.]{0,40}μηνισκ', 'латералния менискус', 'латерален менискус', 'външния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc')}
OA_EVIDENCE = _rx('osteoarthrit', '\\barthros', '\\bgonarthros', '\\bosteoarthros', 'chondropath', 'chondromalac', 'condropat', 'condromalac', '\\bchondros', '\\bchondrosis\\b', 'chondral (loss|defect|ulcer|thinning|injury|fissur|wear)', 'cartilage (loss|thinning|defect|fissur|wear|damage|heterogeneity|irregularit)', '(loss|thinning|fissur|defect|ulcer|erosion|denudation) of[^.]{0,20}cartilage', 'articular cartilage[^.]{0,30}(loss|thin|fissur|defect|erosion|wear|irregular)', 'osteophyt', 'osteofit', 'osteofyt', 'osteofito', 'osteophyten', 'spurring', 'joint space narrowing', 'pinzamiento articular', 'reduced joint space', 'kikirdak kayb', 'kikirdak incelme', 'kondropati', 'kondral', 'kikirdak dejener', 'eklem aralig\\w* daral', 'eklem mesafesi daral', 'kikirdak kalinlig\\w* azal', 'kraakbeen', 'gonartrose', 'artrose', '\\bknorpel', 'arthrose', 'gonarthrose', 'hrskavic', 'hondromalac', 'artroz', 'osteoartrit', 'artrotsk', 'artrotick', '\\boa promjen', '\\boa\\b', 'degenerativne promjene hrskav', 'χονδρ[^ ]*παθ', 'αρθριτ', 'αρθρωσ', 'οστεοφυτ', 'χονδρομαλακ', 'αρθρικου χονδρου', 'εξαλειψη του αρθρικου χονδρου', 'διαβρωση του αρθρικου χονδρ', 'λεπτυνση[^.]{0,30}χονδρ', 'φθορα[^.]{0,20}χονδρ', 'артроз', 'хондропат', 'остеофит', 'хрущял[^.]{0,40}(изтън|увред|дефект|липс)', 'изтъняване[^.]{0,30}хрущял', 'хондромалац', 'ulcera[s]? condral', 'cartilago[^.]{0,25}(perdida|adelgaz)', 'icrs grade', 'icrs\\b', 'outerbridge', '\\bdenudation\\b', 'denudacij', 'erozivne promjene', '\\berosion of[^.]{0,20}cartilage', 'kraakbeenlijden', 'kraakbeenverlies')
TF_SITE = _rx('compartment', 'compartimento', 'compartiment', 'kompartman', 'kompartiment', 'kompartment', 'odjelj', 'διαμερισμα', 'компартм', '\\bотдел', 'femorotibial', 'tibiofemoral', 'femoro tibial', 'femorotibiaal', 'femorotibijaln', 'феморотибиал', '\\bft zglob', 'tibiofemoraln', 'condyle', 'condilo', 'kondyl', 'kondil', 'condyl', 'κονδυλ', 'кондил', '\\bplateau', '\\bplato\\b', 'platillo', 'meseta', 'плато', 'tibiaplateau', 'tibijaln\\w* plato', 'tibyal plato', 'tibia plato', 'κνημιαι', 'μηριαι', 'weightbearing', 'weightbaring', 'zona de carga', 'dragende deel', 'agirlik tasiyan', '\\bfemur\\b', '\\btibia\\b', '\\bfemoral\\b', '\\btibial\\b', '\\bfemura\\b', '\\btibije\\b', '\\bmesarthrio\\b', 'μεσαρθριο')
PF_SITE = _rx('patellofemoral', 'femoropatellar', 'femoropatelar', 'patelofemoral', 'retropatellar', 'retrorotulian', 'trochlea', 'troclea', 'troklea', 'trochlear', 'trohlej', 'τροχιλ', '\\bpatella', '\\bpatellar', 'rotulian', '\\brotula\\b', '\\bpatele\\b', 'patellofemoraal', 'femoropatellair', 'επιγονατιδ', 'μηροεπιγονατιδ', 'пател', 'феморопател', 'anterior compartment', 'compartimento anterior', 'prednj\\w* odjeljk', '\\bfp zglob', '\\bpf zglob', '\\bfaset', '\\bfacet', 'patellofemoraln')
SIDE_MEDIAL = _rx('\\bmedial\\w*', '\\bmedyal\\w*', '\\bmedijaln\\w*', '\\bmediaal\\w*', '\\bmediale\\w*', '\\binterno\\b', '\\binterna\\b', '\\binternos\\b', '\\binterne\\b', '\\binnen\\w*', '\\bic\\b', '\\bunutarnj\\w*', '\\bεσω\\w*', '\\bεσωτερικ\\w*', '\\bмедиал\\w*', '\\bвътреш\\w*', '\\bbinnen\\w*', '\\bmediaal\\b', '\\bmediales?\\b')
SIDE_LATERAL = _rx('\\blateral\\w*', '\\bexterno\\b', '\\bexterna\\b', '\\bexternos\\b', '\\bexterne\\b', '\\bdis\\b', '\\blateraln\\w*', '\\baussen\\w*', '\\bbuiten\\w*', '\\bεξω\\w*', '\\bεξωτερικ\\w*', '\\bлатерал\\w*', '\\bвъншн\\w*', '\\bvanjsk\\w*')
SIDE_ANTERIOR = _rx('\\banterior\\w*', '\\bant\\b', '\\bon\\b', '\\bprednj\\w*', '\\bvorder\\w*', '\\bvoorste\\b', '\\bπροσθι\\w*', '\\bпредн\\w*', '\\banteriyor\\w*', '\\bavant\\b', '\\banterieur\\w*')
GLOBAL_OA = _rx('tri ?compartment', 'all three compartment', 'global(ised)? (oa|osteoarthrit)', '\\bgonarthros', '\\bgonartros', '\\bgonarthrose', '\\bgonartrose', 'gonartro', 'goanrtrot', 'gonartrot', 'osteoarthritis of the knee', 'artrosis (de |)(la )?rodilla', 'knee osteoarthrit', '\\bdiz osteoartrit', '\\bgonartroz', 'artroza koljena', 'οστεοαρθριτιδα', 'αρθριτιδα του γονατος', 'εκφυλιστικη οστεοαρθριτ', 'артроза на колянната', 'гонартроз', 'degenerative joint disease', '\\bdjd\\b', 'three compartments', 'compartmens', 'compartments')
DIRECT = {'Effusion': _rx('\\beffusion', 'joint fluid', 'intra ?articular fluid', '\\bhydrops\\b', '\\bhemarthros', '\\bhaemarthros', 'derrame articular', '\\bderrame\\b', 'liquido articular', 'hemartrosis', 'epanchement', 'gewrichtsvocht', '\\bvocht\\b', 'gewrichtseffusie', 'opzetting van suprapatell', 'gelenkerguss', '\\berguss\\b', 'gelenksergu', 'gelenksflussigkeit', 'eklem\\w* ic\\w* sivi', 'efuzyon', 'eklem sivisi', 'eklem mesafesinde sivi', 'sivi (miktari|artisi|birikimi)', 'sivi artis', '\\bsivi\\b[^.]{0,25}artmis', '\\bizljev', '\\bizliv', 'zglobn[^ ]* tekucin', '\\bhidrops\\b', 'αρθρικ[^ ]* υγρ', 'υγρου ενδαρθρικα', 'ενδαρθρικ[^ ]* υγρ', 'ποσοτητα υγρου', 'ενδαρθρικ', 'αρθρικη συλλογη', 'υγρο στην αρθρωση', 'υγρου στην αρθρωση', 'συλλογη υγρου', 'ενθαρθρικ', 'ставен излив', 'излив', 'ставна течност', 'синовиална течност'), 'Synovitis': _rx('synovit', 'sinovit', 'synovial (thickening|proliferation|hypertroph)', 'thicken\\w* synovial', 'hypertroph\\w* of the synovium', 'synoviale? (verdikking|proliferatie)', 'verdikkingen van (het )?synovium', 'synovialitis', 'synovialis(verdickung|proliferation)', 'reizsynovial', 'sinovijalitis', 'sinovitis', 'zadebljanje sinovij', 'proliferacij\\w* sinovij', 'sinovijaln\\w* proliferacij', 'υμενιτιδα', 'συνοβιτιδα', 'υμενικ[^ ]* υπερτροφ', 'αρθρικου υμεν', 'παχυνση[^.]{0,20}υμεν', 'υμενα', 'синовит', 'синовиал[^ ]* (задебел|пролифер)', '\\bpannus\\b', '\\bhoffit', 'sinovyal\\w* (kalinlas|proliferas)', 'sinovyal hipertrof', '\\bartrit\\b', '\\barthritis\\b'), "Baker's": _rx('baker', 'popliteal cyst', 'quiste popliteo', 'quistes popliteos', 'kyste poplite', 'popliteale? cyst', 'poplitealzyste', 'bakerzyste', 'popliteal kist', '\\bbakerova\\b', 'poplitealn[^ ]* cist', 'popliteal\\w* cist', 'κυστη baker', 'πολυχωρη συνοβιακη κυστη', 'κυστη του baker', 'συνοβιακη κυστη', 'κυστη τυπου baker', 'киста на бейкър', 'бейкърова киста', 'поплитеална киста', 'бекеров', 'gastrocnemio ?semimembranos', 'gastrocnemius semimembranosus burs'), 'Contusion': _rx('\\bcontusion', 'bone bruise', 'bone marrow (o?edema|contusion)', 'marrow o?edema', '\\bkontuz', 'medular bone o?edema', 'osseous contusion', 'contusion osea', 'edema oseo', 'edema de medula osea', 'contusiones oseas', 'oedeme osseux', 'contusion osseuse', 'botcontusie', 'botoedeem', 'beenmergoedeem', 'botmergoedeem', 'knochenmarkodem', 'knochenodem', 'knochenmarksodem', 'kontusion', 'kemik kontuzyonu', 'kemik iligi odemi', 'kemik odemi', 'kemik iliginde odem', 'kontuzyonel kemik', 'kemik iligi odemleri', 'kostani edem', 'edem kosti', 'kontuzij', 'kostane srzi[^.]{0,20}edem', 'οστεομυελικ[^ ]* οιδημα', 'οστικο οιδημα', 'μυελικο οιδημα', 'οστικο μωλωπ', 'костномозъчен едем', 'костен едем', 'контузионен', 'костно мозъчен едем'), 'Fracture': _rx('\\bfractur', '\\bfract\\b', '\\bfractura', '\\bfracturas\\b', '\\bfractuur', '\\bbreuk\\b', '\\bfraktur', '\\bbruch\\b', '\\bkirik\\b', '\\bkirigi\\b', '\\bkiri[kg]\\w*', '\\bprijelom', 'impresijsk[^ ]* fraktur', 'impaktcij', 'καταγμα', 'καταγματ', 'фрактур', 'счупван', 'фисур', 'insufficiency fracture', 'stress fracture', 'avulsion fracture', 'subchondral fracture', 'subkondral kiri', 'impaction (fracture|injury)', 'osteochondral (fracture|impaction)', '\\bsegond\\b', 'impactiefractuur', 'subchondrale impression', 'subchondraler? impress')}
DECOY = {'Fracture': _rx('microfractur', '\\bfracture (risk|prophyla)'), "Baker's": _rx('meniscal cyst', 'quiste meniscal', 'parameniscal')}
PAIRED = {'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus'}
OA_TARGETS = ['Medial OA', 'Lateral OA', 'PF OA']
PLURAL_MENISCI = _rx('\\bmenisci\\b', '\\bmeniscos\\b', '\\bmenisques\\b', '\\bmenisken\\b', '\\bmeniskusi\\b', '\\bmenisk\\w*ler\\b', '\\bμηνισκοι\\b', '\\bμηνισκων\\b', '\\bменискуси\\b', '\\bменискусите\\b', '\\bmenisci\\w*\\b')
ANY_SIDE = _rx(SIDE_MEDIAL.pattern, SIDE_LATERAL.pattern)
STEM_MENISCUS = _rx('menisc\\w*', 'menisk\\w*', 'μηνισκ\\w*', 'мениск\\w*')
STEM_CRUCIATE = _rx('cruciate', 'cruzado', 'croise', 'kruisband', 'kreuzband', 'capraz bag\\w*', 'krizn\\w*', 'χιαστ\\w*', 'кръстн\\w*', '\\bacl\\b', '\\blca\\b', '\\bvkb\\b', '\\bocb\\b', '\\bacb\\b')
STEM_COLLATERAL = _rx('collateral\\w*', 'colateral\\w*', 'kollateral\\w*', 'collaterale\\w*', 'kolateraln\\w*', 'yan bag\\w*', 'πλαγι\\w*', 'колатерал\\w*', 'странич\\w*', 'innenband\\w*', 'binnenband\\w*', '\\bmcl\\b', '\\blcm\\b', '\\biyb\\b')
STEM_FRACTURE = _rx('fractur\\w*', 'fraktur\\w*', 'fractuur\\w*', '\\bfract\\b', 'kiri[kgğ]\\w*', 'prijelom\\w*', 'lom kosti', '\\bbreuk\\w*', '\\bbruch\\w*', 'καταγμα\\w*', 'καταγματ\\w*', 'фрактур\\w*', 'счупван\\w*', 'fisur\\w* (osea|oseas|kost)', 'fissur\\w* kost')
POSTERIOR_ONLY = _rx('\\bpcl\\b', '\\blcp\\b', '\\bhkb\\b', '\\bacb\\b', 'posterior cruciate', 'cruzado posterior', 'croise posterieur', 'achterste kruisband', 'hinteres kreuzband', 'arka capraz', 'straznji krizn', 'οπισθι[οα]\\w* χιαστ', 'задна кръстн', 'задната кръстн')
LATERAL_COLL_ONLY = _rx('\\blcl\\b', '\\bfcl\\b', 'lateral collateral', 'fibular collateral', 'colateral lateral', 'colateral externo', 'buitenband', 'aussenband', 'dis yan bag', 'lateralni kolateraln', 'εξω πλαγι', 'латерален колатерал')

In [ ]:
def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int=55):
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False
STEM_RULES = {'ACL': (STEM_CRUCIATE, SIDE_ANTERIOR), 'MCL': (STEM_COLLATERAL, SIDE_MEDIAL), 'Medial Meniscus': (STEM_MENISCUS, SIDE_MEDIAL), 'Lateral Meniscus': (STEM_MENISCUS, SIDE_LATERAL)}

class _Matcher:

    def __init__(self, phrase_rx, stem=None, side=None, window=55):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None
ANAT_MATCH = {t: _Matcher(ANAT[t], *STEM_RULES[t]) for t in PAIRED}
DIRECT_MATCH = {t: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if t == 'Fracture' else rx) for t, rx in DIRECT.items()}
SEV_LOW = _rx('\\bsmall\\b', '\\bminimal\\b', '\\btrace\\b', '\\bmild\\b', '\\bslight\\b', '\\btiny\\b', '\\bscant\\b', '\\bdiscrete\\b', '\\blow ?grade\\b', '\\bincipient\\b', '\\bleve\\b', '\\bminim', '\\bpeque', '\\bfina\\b', '\\bfino\\b', '\\bligero\\b', '\\bescaso\\b', '\\bdiscreto\\b', '\\bhafif\\b', '\\baz miktarda\\b', '\\bsilik\\b', '\\bmanj\\w*', '\\bblago\\b', '\\bdiskretn', '\\bmalo\\b', '\\bpocetn', '\\bgering', '\\bdiskret', '\\bkleine?r?\\b', '\\bwenig\\b', '\\bzarte?\\b', '\\bbeperkte?\\b', '\\bgeringe\\b', '\\bweinig\\b', '\\blichte?\\b', '\\blicht\\b', '\\bηπι', '\\bμικρ', '\\bελαχιστ', '\\bαρχομεν', '\\bминимал', '\\bлек', '\\bмалк', '\\bнеголям')
SEV_HIGH = _rx('\\blarge\\b', '\\bmarked\\b', '\\bmassive\\b', '\\bsevere\\b', '\\bextensive\\b', '\\bmoderate\\b', '\\bgross\\b', '\\bsignificant\\b', '\\babundant\\b', '\\btense\\b', '\\bcomplete\\b', '\\bfull ?thickness\\b', '\\bhigh ?grade\\b', '\\badvanced\\b', '\\bmoderad', '\\bimportante\\b', '\\bsevera?\\b', '\\bmarcad', '\\bcuantios', '\\bespesor total\\b', '\\bcompleta?\\b', '\\bbelirgin\\b', '\\byaygin\\b', '\\bileri\\b', '\\bciddi\\b', '\\bbol\\b', '\\bkomplet', '\\bopsezan\\b', '\\bveliki\\b', '\\bizrazit', '\\bznacajn', '\\bumjeren', '\\buznapredoval', '\\bpotpun', '\\bkompleksn', '\\bausgepragt', '\\bdeutlich', '\\bmassiv', '\\bmassig', '\\bgross', '\\buitgebreid', '\\bgevorderd', '\\bveel\\b', '\\bmatige?\\b', '\\bvolledig', '\\bμετρι', '\\bμεγαλ', '\\bεκτεταμεν', '\\bευμεγεθ', '\\bσοβαρ', '\\bπληρη', '\\bголям', '\\bизразен', '\\bзначим', '\\bумерен', '\\bобилен', '\\bпълн')
GRADE_HIGH = re.compile('grade?[ao]?\\s*(3|4|iii|iv)\\b|icrs grade (iii|iv|3|4)|stupnja iv|stupnja iii|\\bgrado (3|4)\\b|\\bgrad (3|4)\\b|\\bgrade (3|4)\\b')
DEGENERATIVE_MARROW = _rx('subchondral', 'subcondral', 'subkondral', 'supkondraln', 'subchondraln', 'υποχονδρι', 'υπαρθρικ', 'субхондрал', 'subchondrale?', 'subartikuler', '\\bcyst', '\\bquist', '\\bzyste\\b', '\\bcistic', 'reactive', 'reactivo', 'degenerative', 'degenerativ', 'reaktiv', '\\bcisti\\b')
TRAUMA = _rx('\\bbruise\\b', '\\bcontusion', '\\bkontuz', '\\btrauma', '\\bimpaction\\b', '\\bpivot shift\\b', '\\bkissing\\b', '\\bacute\\b', '\\bagudo\\b', '\\bakut', '\\bpivot kaymasi\\b', '\\bcontusion osseuse\\b', '\\bbone bruise\\b', '\\bbotcontusie\\b', '\\bконтузион', '\\bμωλωπ', '\\bkontuzij', '\\bimpaktcij', '\\bimpakcij', '\\bfall\\b', '\\binjury\\b', '\\bimpression\\b')
SYNOVIAL_PROXY = _rx('bursit', 'burzit', '\\bbursa\\b[^.]{0,30}(fluid|distend|sivi|tekucin|opzetting)', 'suprapatellar (bursitis|effusion|recess)', 'suprapatellar bursa', 'suprapatellar bursada', 'suprapatelarno', 'suprapatellaire recessus', 'hoffa', 'hoffit', 'plica', 'plika', 'πλικα', 'fat pad[^.]{0,20}(edema|oedema)', 'kapsul', 'capsul', 'καψ', 'капсул', '\\bpannus\\b', '\\bsinov', '\\bsynov')

def _polarity(clause: str, span=None) -> str:
    if UNCERTAIN.search(clause):
        return 'uncertain'
    if span is None or not FEATURES['directional_negation']:
        if NEGATION.search(clause):
            return 'negative'
    elif _negated(clause, span[0], span[1]):
        return 'negative'
    if NORMALITY.search(clause):
        if TEAR.search(clause) or GRADE_HIGH.search(clause):
            return 'positive'
        return 'negative'
    return 'positive'

def _severity(clause: str) -> float:
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and (not low):
        return 1.0
    if low and (not high):
        return 0.45
    if high and low:
        return 0.8
    return 0.75

def _grade(n_pos, n_neg, n_unc, best):
    if n_pos or n_unc:
        score = min(0.97, 0.5 + 0.45 * best + 0.015 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.2 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = (0.28, 0.05)
    return (score, conf)

def _paired_weight(clause: str, meniscus: bool) -> float:
    g = _grade_of(clause) if FEATURES['graded_pathology'] else None
    tear = TEAR.search(clause) is not None
    if meniscus:
        if tear:
            base = 1.0
        elif g is not None:
            base = 0.95 if g >= 3 else 0.3
        elif DEGEN.search(clause):
            base = 0.35
        else:
            base = 0.45
    elif tear:
        base = 1.0
    elif g is not None:
        base = 0.85 if g >= 2 else 0.3
    elif DEGEN.search(clause):
        base = 0.4
    else:
        base = 0.55
    if SEV_HIGH.search(clause) and (not SEV_LOW.search(clause)):
        base = min(1.0, base * 1.2)
    elif SEV_LOW.search(clause) and (not SEV_HIGH.search(clause)):
        base *= 0.7
    return base

def _score_paired(cls, tgt):
    anat_rx = ANAT_MATCH[tgt]
    path_rx = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)
    meniscus = 'Meniscus' in tgt
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        hit = anat_rx.search(c)
        if hit is None and meniscus and PLURAL_MENISCI.search(c) and (not ANY_SIDE.search(c)):
            hit = PLURAL_MENISCI.search(c)
        if hit is None:
            continue
        pm = path_rx.search(c)
        if pm is None and _grade_of(c) is None:
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        span = (pm.start(), pm.end()) if pm is not None else None
        pol = _polarity(c, span)
        if pol == 'positive':
            n_pos += 1
            best = max(best, _paired_weight(c, meniscus))
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.45 * _paired_weight(c, meniscus))
    s, cf = _grade(n_pos, n_neg, n_unc, best)
    return (s, cf, n_pos, n_neg)

def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None, context_bonus=None):
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and (not path_rx.search(c)):
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        pol = _polarity(c, (m.start(), m.end()))
        if pol == 'positive':
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.3)
    s, c = _grade(n_pos, n_neg, n_unc, best)
    return (s, c, n_pos, n_neg)

def _score_oa(cls):
    acc = {t: {'pos': 0, 'neg': 0, 'unc': 0, 'best': 0.0} for t in OA_TARGETS}
    g_pos, g_neg, g_best = (0, 0, 0.0)
    for c in cls:
        m = OA_EVIDENCE.search(c)
        if not m:
            continue
        pol = _polarity(c, (m.start(), m.end()))
        sev = _severity(c)
        tf_med = _near(c, TF_SITE, SIDE_MEDIAL, 45)
        tf_lat = _near(c, TF_SITE, SIDE_LATERAL, 45)
        pf = PF_SITE.search(c) is not None
        hits = []
        if tf_med:
            hits.append('Medial OA')
        if tf_lat:
            hits.append('Lateral OA')
        if pf:
            hits.append('PF OA')
        if not hits:
            if pol == 'positive':
                g_pos += 1
                g_best = max(g_best, sev if GLOBAL_OA.search(c) else sev * 0.7)
            elif pol == 'negative':
                g_neg += 1
            continue
        for t in hits:
            if pol == 'positive':
                acc[t]['pos'] += 1
                acc[t]['best'] = max(acc[t]['best'], sev)
            elif pol == 'negative':
                acc[t]['neg'] += 1
            else:
                acc[t]['unc'] += 1
                acc[t]['best'] = max(acc[t]['best'], 0.3)
    out = {}
    for t in OA_TARGETS:
        a = acc[t]
        pos, neg, unc, best = (a['pos'], a['neg'], a['unc'], a['best'])
        if not (pos or unc) and g_pos and FEATURES['oa_inherit']:
            if neg:
                score, conf = _grade(0, neg, 0, 0.0)
                score = max(score, 0.35)
                conf *= 0.7
            else:
                score, conf = _grade(g_pos, 0, 0, g_best * 0.92)
                conf *= 0.75
        else:
            score, conf = _grade(pos, neg + g_neg, unc, best)
        out[t] = (score, conf, pos, neg)
    return out

def extract(report: str) -> dict:
    cls = clauses(report)
    out = {}
    for tgt in PAIRED:
        s, c, npos, nneg = _score_paired(cls, tgt)
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt, (s, c, npos, nneg) in _score_oa(cls).items():
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt in ('Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'):
        if tgt == 'Contusion':
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt), context_penalty=DEGENERATIVE_MARROW, context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    if FEATURES['synovitis_backoff'] and out['Synovitis__npos'] == 0 and (out['Synovitis__nneg'] == 0):
        proxy = sum((1 for c in cls if SYNOVIAL_PROXY.search(c) and _polarity(c) == 'positive'))
        eff = out['Effusion']
        prior = 0.3 + 0.3 * max(0.0, (eff - 0.5) / 0.45) + 0.06 * min(proxy, 3)
        out['Synovitis'] = min(0.72, prior)
        out['Synovitis__conf'] = 0.18
    return out

In [ ]:
# Run the lexicon over every training report.
_t = time.time()
LAB = pd.DataFrame([extract(r) for r in train["Report"].fillna("")])
LAB["StudyInstanceUID"] = train["StudyInstanceUID"].values
LAB = LAB.set_index("StudyInstanceUID")
STAGE_SECS["report lexicon"] = round(time.time() - _t, 1)
log("read %d reports in %.1fs" % (len(LAB), time.time() - _t))

# Binary target + per-study confidence used as the TM sample weight.
Y_REPORT = (LAB[LABELS] > 0.5).astype(int)
W_REPORT = LAB[[t + "__conf" for t in LABELS]].copy()
W_REPORT.columns = LABELS

# The 58 expert rows win where they exist, at full weight.
GOLD = train.dropna(subset=LABELS).set_index("StudyInstanceUID")[LABELS]
if len(GOLD):
    Y_REPORT.loc[GOLD.index, LABELS] = GOLD.values.astype(int)
    W_REPORT.loc[GOLD.index, LABELS] = 1.0

SILENT = pd.DataFrame(
    {t: ((LAB[t + "__npos"] == 0) & (LAB[t + "__nneg"] == 0)).values for t in LABELS},
    index=LAB.index)

In [ ]:
from sklearn.metrics import roc_auc_score

def agreement_table():
    if not len(GOLD):
        return pd.DataFrame()
    g = LAB.loc[GOLD.index]
    rows = []
    for t in LABELS:
        y = GOLD[t].values.astype(int)
        if len(set(y)) < 2:
            rows.append((t, np.nan, int(y.sum()), int((1 - y).sum()), SILENT[t].mean()))
            continue
        rows.append((t, roc_auc_score(y, g[t].values), int(y.sum()),
                     int((1 - y).sum()), SILENT[t].mean()))
    return pd.DataFrame(rows, columns=["finding", "agreement AUC", "npos", "nneg", "silence rate"])

# Which language a report is in, so the feedback report can show where the lexicon
# goes silent -- a finding silent in one language and not another is a lexicon gap.
# (function from the v41 notebook)
_SCRIPT = {"el": re.compile("[Ͱ-Ͽ]"), "bg/ru": re.compile("[Ѐ-ӿ]")}
_STOP = {
    "en": r"\b(the|and|is|with|there is|normal)\b",
    "es": r"\b(del|los|las|con|sin|senal|rodilla|hallazgos|tecnica|resultados|impresion|menisco|rotura)\b",
    "fr": r"\b(des|les|avec|sans|genou|aucune)\b",
    "nl": r"\b(van|het|een|geen|met|voorste|knie)\b",
    "de": r"\b(der|die|und|mit|ohne|kein|keine|nachweis)\b",
    "tr": r"\b(ve|ile|izlenmistir|mevcut|normaldir|diz|bulgular)\b",
    "hr": r"\b(se|te|uz|bez|prikaz|uredan|koljena|meniska)\b",
}
_STOP = {k: re.compile(v) for k, v in _STOP.items()}

def guess_language(report):
    n = normalize(report)
    for tag, rx in _SCRIPT.items():
        if rx.search(n):
            return tag
    score = {k: len(rx.findall(n)) for k, rx in _STOP.items()}
    best = max(score, key=score.get)
    return best if score[best] >= 2 else "?"

LANG = pd.Series([guess_language(r) for r in train["Report"].fillna("")],
                 index=train["StudyInstanceUID"]) if "Report" in train.columns else \
       pd.Series("?", index=train["StudyInstanceUID"])
SILENCE_BY_LANG = (SILENT.groupby(LANG.reindex(SILENT.index).values).mean() * 100).round(1)
SILENCE_BY_LANG.insert(0, "n_studies", LANG.value_counts().reindex(SILENCE_BY_LANG.index).values)

AGREE = agreement_table()
if len(AGREE):
    print(AGREE.round(3).to_string(index=False))
    print("\nmacro agreement AUC: %.4f   (n = %d annotated studies, so read it as a smoke test,"
          " not a precise number)" % (AGREE["agreement AUC"].mean(), len(GOLD)))
print("\nderived positive rate vs annotated positive rate:")
print(pd.DataFrame({"derived": Y_REPORT.mean().round(3),
                    "annotated": GOLD.mean().round(3) if len(GOLD) else np.nan,
                    "mean confidence": W_REPORT.mean().round(3)}).to_string())

## 3. The Tsetlin Machine, rewritten as matrix products

Same algorithm as v2 — the pyTsetlinMachine v3 binary TM: 2F literals, 8-bit thermometer TA states
with inclusion at the midpoint, even/odd clause parity, `class_sum`-gated Type Ia / Ib / II feedback,
`max_included_literals` to keep clauses sparse. Two changes:

1. **Feedback is accumulated over a mini-batch and applied once**, which turns the per-clause Python
   loop into two float32 GEMMs per class per batch (`gate.T @ L` for the literals seen, `M @ L.T` for
   the clause outputs). This is the same trade the multi-threaded CAIR implementations make when
   several threads write one shared state array.
2. **Clause weights are on by default**, and stochastic decrements use their batch expectation
   rather than one coin flip per literal — after aggregating 64 samples the coin flips have already
   averaged out, and drawing 3 M gaussians per batch cost more than the GEMMs did.

`sample_weight` scales the feedback gate, which is how the report confidence from §2 enters: a
study the lexicon was unsure about updates the machine less often than one it was sure about.

In [ ]:
class BatchTsetlinMachine:
    """Binary TM whose clause evaluation and feedback accumulation are float32 GEMMs."""

    def __init__(self, clauses=200, T=1000, s=5.0, epochs=8, batch=64,
                 max_included_literals=32, weighted=True, boost=True,
                 state_bits=8, seed=0, noisy_feedback=False):
        self.clauses = int(clauses)
        self.T = float(T)
        self.s = float(s)
        self.epochs = int(epochs)
        self.batch = int(batch)
        self.max_included = max_included_literals
        self.weighted = weighted
        self.boost = boost
        self.seed = seed
        self.noisy_feedback = noisy_feedback
        self.INC_THR = 2 ** (state_bits - 1)
        self.STATE_MAX = 2 ** state_bits - 1

    def _init(self, n_features):
        self.F = int(n_features)
        self.LIT = 2 * self.F
        self._rng = np.random.default_rng(self.seed)
        # Every TA one step below inclusion: all clauses start empty and fire on
        # everything during training, which is what bootstraps learning.
        self.states = np.full((2, self.clauses, self.LIT), self.INC_THR - 1, np.int16)
        self.weights = np.ones((2, self.clauses), np.float32)
        self.sign = np.where(np.arange(self.clauses) % 2 == 0, 1.0, -1.0).astype(np.float32)
        if self.max_included is None:
            self.max_included = self.LIT

    def _lits(self, Xb):
        L = np.empty((len(Xb), self.LIT), np.float32)
        L[:, :self.F] = Xb                       # casts uint8 -> float32
        np.subtract(1.0, L[:, :self.F], out=L[:, self.F:])
        return L

    def _stoch(self, n, p):
        """Binomial(n, p) elementwise; the batch already averaged the coin flips."""
        if not self.noisy_feedback:
            return n * p
        sd = np.sqrt(np.maximum(n, 0.0) * p * (1.0 - p))
        return np.clip(n * p + sd * self._rng.standard_normal(n.shape).astype(np.float32), 0.0, n)

    def fit(self, X, y, sample_weight=None):
        y = np.asarray(y).astype(np.int8)
        w = (np.ones(len(X), np.float32) if sample_weight is None
             else np.clip(np.asarray(sample_weight, np.float32), 0.0, 1.0))
        self._init(X.shape[1])
        p_lit = 1.0 / self.s
        for _ in range(self.epochs):
            order = self._rng.permutation(len(X))
            for b0 in range(0, len(X), self.batch):
                sel = order[b0:b0 + self.batch]
                self._update(X[sel], y[sel], w[sel], p_lit)
        return self

    def _update(self, Xb, yb, wb, p_lit):
        L = self._lits(Xb)
        Lc = 1.0 - L
        B = len(L)
        for c in (0, 1):
            M = self.states[c] >= self.INC_THR                      # (C, LIT)
            Mf = M.astype(np.float32)
            counts = Mf.sum(1)
            out = (L @ Mf.T) >= counts[None, :] - 0.5               # (B, C); empty clauses fire
            cs = np.clip((out * (self.sign * self.weights[c])[None, :]).sum(1), -self.T, self.T)

            target = (yb == c)
            gate_p = (self.T + np.where(target, -1.0, 1.0) * cs) / (2.0 * self.T)
            gate = self._rng.random((B, self.clauses), dtype=np.float32) <= (gate_p * wb)[:, None]

            type_i = (self.sign[None, :] > 0) == target[:, None]     # polarity matches the target
            ia = (gate & type_i & out & (counts <= self.max_included)[None, :]).astype(np.float32)
            ib = (gate & type_i & ~out).astype(np.float32)
            ii = (gate & ~type_i & out).astype(np.float32)

            d = None
            n_ia = ia.sum(0)
            if n_ia.any():                                           # Type Ia: reinforce the truth
                n_true = ia.T @ L
                d = (n_true if self.boost else self._stoch(n_true, p_lit))
                d = d - self._stoch(n_ia[:, None] - n_true, p_lit)
                if self.weighted:
                    self.weights[c] += n_ia
            n_ib = ib.sum(0)
            if n_ib.any():                                           # Type Ib: forget, uniformly
                dec = (self._stoch(np.broadcast_to(n_ib[:, None], (self.clauses, self.LIT)), p_lit)
                       if self.noisy_feedback else (n_ib * p_lit)[:, None])
                d = -dec if d is None else d - dec
            n_ii = ii.sum(0)
            if n_ii.any():                                           # Type II: include what would
                e = (ii.T @ Lc) * (~M)                               # falsify the clause
                d = e if d is None else d + e
                if self.weighted:
                    self.weights[c] = np.maximum(1.0, self.weights[c] - n_ii)
            if d is None:
                continue
            np.clip(self.states[c] + np.rint(d).astype(np.int16), 0, self.STATE_MAX,
                    out=self.states[c])

    def class_sums(self, X, batch=512):
        out = np.zeros((len(X), 2), np.float32)
        banks = []
        for c in (0, 1):
            M = (self.states[c] >= self.INC_THR).astype(np.float32)
            banks.append((M, M.sum(1)))
        for b0 in range(0, len(X), batch):
            L = self._lits(X[b0:b0 + batch])
            for c in (0, 1):
                M, counts = banks[c]
                # empty clauses are suppressed at predict time, unlike during training
                fired = ((L @ M.T) >= counts[None, :] - 0.5) & (counts[None, :] > 0)
                out[b0:b0 + len(L), c] = (fired * (self.sign * self.weights[c])[None, :]).sum(1)
        return out

    def decision_function(self, X):
        cs = self.class_sums(X)
        return cs[:, 1] - cs[:, 0]

    def n_included(self):
        return int((self.states >= self.INC_THR).sum())

In [ ]:
# Sanity check + the head-to-head against v2's TM quoted at the top of this notebook.
# Runs anywhere, needs no competition data.
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

_d = load_digits()
_img = _d.images / 16.0
_X = np.stack([_img >= t for t in np.linspace(0.15, 0.85, 4)], -1).reshape(len(_img), -1).astype(np.uint8)
_y = (_d.target % 2 == 0).astype(int)
_Xtr, _Xte, _ytr, _yte = train_test_split(_X, _y, test_size=0.3, random_state=0, stratify=_y)

for tag, kw in [("v2 hyper-parameters, unweighted clauses",
                 dict(clauses=150, T=800, s=3.0, epochs=10, max_included_literals=8, weighted=False)),
                ("v2 hyper-parameters, weighted clauses",
                 dict(clauses=150, T=800, s=3.0, epochs=10, max_included_literals=8, weighted=True)),
                ("v4 defaults",
                 dict(clauses=TM_CLAUSES, T=TM_T, s=TM_S, epochs=TM_EPOCHS,
                      max_included_literals=TM_MAX_INCLUDED, weighted=TM_WEIGHTED))]:
    _t = time.time()
    _tm = BatchTsetlinMachine(batch=TM_BATCH, seed=1, **kw).fit(_Xtr, _ytr)
    _auc = roc_auc_score(_yte, _tm.decision_function(_Xte))
    print("digits even-vs-odd  AUC %.3f  fit %4.1fs   %s" % (_auc, time.time() - _t, tag))
print("\nfor reference, v2's TM on this same task: AUC 0.535 after 10 epochs (40 s), 0.919 after 25.")
assert _auc > 0.90, "the TM is not learning an easy task; stop here rather than train on knees"

## 4. Slots — which series a finding is actually visible on

v2 picked one series per anatomical plane. But a joint effusion on a T1 sagittal is nearly invisible
and obvious on a fat-suppressed fluid-sensitive one, and `train_series.csv` already tells us which is
which (`Anatomical_Plane`, `Fluid_Sensitive`, `Fat_Suppression`) — no DICOM header pass needed.

So, as in v41, a study is described by **slots**: (plane × fluid-sensitive × fat-suppressed). Each
finding names the slots it is worth looking at, in order, and `SLOTS_PER_TARGET` of them are tried.
Fallback is graded — relax fat suppression, then fluid sensitivity, then take any series in the
plane — so a study missing the ideal sequence still contributes.

In [ ]:
# name, plane, fluid-sensitive, fat-suppressed
SLOTS = [
    ("SAG_FS", "Sagittal", True,  True),    # fluid, fat-sat: effusion, oedema, synovitis
    ("SAG_PD", "Sagittal", True,  False),   # PD/T2 no fat-sat: menisci, cruciates
    ("COR_FS", "Coronal",  True,  True),    # collaterals, compartments, marrow
    ("AX_FS",  "Axial",    True,  True),    # patellofemoral, Baker's
]
SLOT_NAMES = [s[0] for s in SLOTS]
SLOT_PLANE = {s[0]: s[1] for s in SLOTS}

# Which slot to read a finding off, best first.
SLOT_PRIOR = {
    "ACL":              ["SAG_PD", "SAG_FS"],
    "MCL":              ["COR_FS", "SAG_PD"],
    "Medial Meniscus":  ["SAG_PD", "COR_FS"],
    "Lateral Meniscus": ["SAG_PD", "COR_FS"],
    "Medial OA":        ["COR_FS", "SAG_PD"],
    "Lateral OA":       ["COR_FS", "SAG_PD"],
    "PF OA":            ["AX_FS",  "SAG_PD"],
    "Effusion":         ["SAG_FS", "AX_FS"],
    "Synovitis":        ["SAG_FS", "AX_FS"],
    "Baker's":          ["AX_FS",  "SAG_FS"],
    "Contusion":        ["COR_FS", "SAG_FS"],
    "Fracture":         ["COR_FS", "SAG_FS"],
}

def _as_bool(col):
    if col.dtype == bool:
        return col
    return col.astype(str).str.strip().str.lower().isin(["1", "true", "yes", "y", "t"])

def slot_assignment(series_df, split):
    """{study: {slot: series_dir}} — one series per slot, graded fallback, most slices wins."""
    df = series_df.copy()
    df["_plane"] = df["Anatomical_Plane"].astype(str).str.strip().str.title()
    df["_fluid"] = _as_bool(df["Fluid_Sensitive"]) if "Fluid_Sensitive" in df else False
    df["_fs"]    = _as_bool(df["Fat_Suppression"]) if "Fat_Suppression" in df else False
    root = ROOT / f"{split}_series"

    n_files = {}
    def count(args):
        st, sid = args
        d = root / st / sid
        try:
            return (st, sid, sum(1 for e in os.scandir(d) if e.name.endswith(".dcm")))
        except OSError:
            return (st, sid, 0)
    pairs = list(zip(df["StudyInstanceUID"], df["SeriesInstanceUID"]))
    with ThreadPoolExecutor(max_workers=IO_THREADS) as pool:
        for st, sid, n in pool.map(count, pairs):
            n_files[(st, sid)] = n
    df["_n"] = [n_files.get(k, 0) for k in pairs]
    df = df[df["_n"] > 0]

    out = {}
    for st, g in df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            in_plane = g[g["_plane"] == plane]
            for cand in (in_plane[(in_plane["_fluid"] == fluid) & (in_plane["_fs"] == fs)],
                         in_plane[in_plane["_fluid"] == fluid],
                         in_plane):
                if len(cand):
                    r = cand.sort_values("_n", ascending=False).iloc[0]
                    chosen[name] = str(r["SeriesInstanceUID"])
                    break
        if chosen:
            out[st] = chosen
    return out

## 5. Pixels — ordering, physical crop, laterality, several slices

Four things v2 did not do, all of them cheap and all of them ported from v41:

* **Order the stack by geometry.** Files are named by SOPInstanceUID, so file order is arbitrary.
  Slices are sorted by their projection on the slice normal (`ImagePositionPatient · (r × c)`),
  falling back to `InstanceNumber`, then to a natural filename sort.
* **Take several slices, not one.** `N_SLICE` slices spread over the middle half of the stack
  (`SLICE_BAND`). Each is an independent training sample carrying the study's label, and the study
  score is pooled back from its slices — which is what lets a finding that occupies three slices be
  found at all. The pooling (max / top-2 / mean) is chosen per finding by OOF in §6, the same
  device v41 uses per target in its TTA.
* **Crop by millimetres, not pixels.** `CROP_MM / PixelSpacing` gives the pixel width of a fixed
  160 mm field, so the joint is the same size in every study before downsampling.
* **Normalise laterality.** Right knees are mirrored versions of left ones. The DICOM
  `Laterality` / `ImageLaterality` tag, or failing that the sign of the patient x coordinate,
  decides; right knees get their slice order reversed on sagittal and a horizontal flip on
  coronal/axial. Without this the machine sees each finding in two mirrored forms.

In [ ]:
def _rows_area(a, size):
    n = a.shape[0]
    idx = (np.arange(size) * n) // size
    cnt = np.diff(np.append(idx, n)).astype(np.float32)
    return np.add.reduceat(a, idx, axis=0) / cnt[:, None]

def resize_area(img, size):
    """Alias-free area-average resize (numpy only)."""
    h, w = img.shape
    if h < size or w < size:
        r = int(np.ceil(max(size / h, size / w)))
        img = np.repeat(np.repeat(img, r, 0), r, 1)
    return _rows_area(_rows_area(img, size).T, size).T

def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(name)))

_ORDER_TAGS = ["InstanceNumber", "ImagePositionPatient", "ImageOrientationPatient",
               "PixelSpacing", "Laterality", "ImageLaterality"]

def read_slot(series_dir, n_slice=None, out_size=None):
    """-> (uint8 stack (n_slice, out, out), laterality or None) for one series."""
    import pydicom
    n_slice = N_SLICE if n_slice is None else n_slice
    out_size = CACHE_IMG if out_size is None else out_size
    try:
        files = sorted(e.name for e in os.scandir(series_dir) if e.name.endswith(".dcm"))
    except OSError:
        return None, None
    if not files:
        return None, None

    keys, px, lat, xs, normal = [], None, None, [], None
    for f in files:
        proj = inst = None
        try:
            h = pydicom.dcmread(os.path.join(series_dir, f), stop_before_pixels=True,
                                force=True, specific_tags=_ORDER_TAGS)
            ipp = getattr(h, "ImagePositionPatient", None)
            iop = getattr(h, "ImageOrientationPatient", None)
            if ipp is not None and len(ipp) >= 3:
                ipp = np.asarray(ipp[:3], float)
                xs.append(float(ipp[0]))
                if normal is None and iop is not None and len(iop) >= 6:
                    v = np.asarray(iop[:6], float)
                    normal = np.cross(v[:3], v[3:6])
                if normal is not None:
                    proj = float(np.dot(ipp, normal))
            n = getattr(h, "InstanceNumber", None)
            if n is not None:
                inst = float(n)
            if px is None:
                ps = getattr(h, "PixelSpacing", None)
                if ps is not None and len(ps) >= 1:
                    px = float(ps[0])
            if lat is None:
                for tag in ("ImageLaterality", "Laterality"):
                    v = str(getattr(h, tag, "") or "").strip().upper()
                    if v[:1] in ("L", "R"):
                        lat = v[0]
                        break
        except Exception:
            pass
        keys.append((proj, inst, f))

    placed = sum(k[0] is not None for k in keys)
    if placed >= max(2, int(0.8 * len(keys))):
        keys.sort(key=lambda k: (k[0] if k[0] is not None else np.inf, _natural_key(k[2])))
    elif sum(k[1] is not None for k in keys) >= max(2, int(0.8 * len(keys))):
        keys.sort(key=lambda k: (k[1] if k[1] is not None else np.inf, _natural_key(k[2])))
    else:
        keys.sort(key=lambda k: _natural_key(k[2]))
    ordered = [k[2] for k in keys]

    if lat is None and xs:
        m = float(np.median(xs))
        lat = None if abs(m) < 20.0 else ("R" if m < 0 else "L")

    n = len(ordered)
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, max(hi, lo), n_slice).astype(int))
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            # Some DICOMs use JPEG2000 / JPEG-LS, which stock pydicom cannot decode.
            ds = pydicom.dcmread(os.path.join(series_dir, ordered[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            if a.ndim == 3:
                a = a[a.shape[0] // 2]
            a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
        except Exception:
            a = None
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if not got:
        return None, lat
    shp = planes[got[0]].shape
    planes = [planes[min(got, key=lambda j: abs(j - k))] if p is None else p
              for k, p in enumerate(planes)]
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    if px and np.isfinite(px) and px > 0:                    # physical crop
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx, half = h // 2, w // 2, want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)
    stack = np.stack([resize_area(v, out_size) for v in vol])
    return (stack * 255).round().clip(0, 255).astype(np.uint8), lat

def normalise_laterality(stack, plane, lat):
    """Map right knees onto the left-knee convention."""
    if lat != "R":
        return stack
    if plane in ("Coronal", "Axial"):
        return stack[:, :, ::-1].copy()
    return stack[::-1].copy()          # sagittal: medial<->lateral is the slice axis

In [ ]:
DECODE_STATS = {}

def build_cache(uids, assign, split, tag):
    """-> (uids kept, uint8 cache (n, n_slot, N_SLICE, CACHE_IMG, CACHE_IMG), mask (n, n_slot))"""
    t_cache = time.time()
    uids = [u for u in uids if u in assign]
    sidx = {u: i for i, u in enumerate(uids)}
    cache = np.zeros((len(uids), len(SLOTS), N_SLICE, CACHE_IMG, CACHE_IMG), np.uint8)
    mask = np.zeros((len(uids), len(SLOTS)), np.float32)
    root = ROOT / f"{split}_series"
    jobs = [(u, k, name) for u in uids for k, name in enumerate(SLOT_NAMES)
            if name in assign[u]]
    log("%s: %d studies, %d slot-series to decode (%.2f GB cache)"
        % (tag, len(uids), len(jobs), cache.nbytes / 1024 ** 3))

    def run(job):
        u, k, name = job
        return job, read_slot(root / u / assign[u][name])

    # Chunked, because ThreadPoolExecutor.map submits every job up front: breaking
    # out of it would still wait for all of them. One chunk is the budget granularity.
    CHUNK, done, stopped = 512, 0, False
    with ThreadPoolExecutor(max_workers=IO_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            for (u, k, name), (stack, lat) in pool.map(run, jobs[c0:c0 + CHUNK]):
                done += 1
                if stack is not None:
                    cache[sidx[u], k] = normalise_laterality(stack, SLOT_PLANE[name], lat)
                    mask[sidx[u], k] = 1.0
            if done % 2048 < CHUNK:
                log("  %s %d/%d" % (tag, done, len(jobs)))
            if time_left() < 0.35 * TIME_BUDGET:
                stopped = True
                break
    if stopped:
        log("  %s: decode budget reached at %d/%d; the rest stay empty" % (tag, done, len(jobs)))
    log("%s: %d/%d slots filled" % (tag, int(mask.sum()), len(jobs)))
    DECODE_STATS[tag] = dict(studies=len(uids), jobs=len(jobs), attempted=done,
                             filled=int(mask.sum()), truncated=stopped,
                             secs=round(time.time() - t_cache, 1))
    gc.collect()
    return uids, cache, mask

## 6. Binary feature maps

Unchanged in spirit from v2/v3 — each preprocessing method turns a slice into its own bit vector and
gets its own machine, which is what makes this a *composite*. Two differences: the maps run at
`TM_IMG` (downsampled from the 64 px cache, so resolution is a knob not a rebuild), and every map is
contrast-normalised per slice first, so a bright fat-sat sequence and a dark T1 produce comparable
bits.

In [ ]:
def _norm01(img):
    lo, hi = np.percentile(img, [2, 98])
    return np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) if hi > lo else np.zeros_like(img)

def otsu_binary(img):
    hist, edges = np.histogram(img, bins=64, range=(0.0, 1.0))
    centers = 0.5 * (edges[:-1] + edges[1:])
    w = np.cumsum(hist); wb = hist.sum() - w
    m = np.cumsum(hist * centers); mg = m[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        between = w * wb * (m / np.where(w == 0, 1, w) - (mg - m) / np.where(wb == 0, 1, wb)) ** 2
    return img >= centers[np.nanargmax(between)]

def box_mean(img, k=5):
    H, W = img.shape
    ii = np.zeros((H + 1, W + 1))
    ii[1:, 1:] = np.cumsum(np.cumsum(img, 0), 1)
    half = k // 2
    bot = np.minimum(np.arange(H) + half + 1, H); top = np.maximum(np.arange(H) - half, 0)
    right = np.minimum(np.arange(W) + half + 1, W); left = np.maximum(np.arange(W) - half, 0)
    sums = (ii[np.ix_(bot, right)] - ii[np.ix_(top, right)]
            - ii[np.ix_(bot, left)] + ii[np.ix_(top, left)])
    cnt = (bot - top)[:, None] * (right - left)[None, :]
    return sums / np.maximum(cnt, 1)

def adaptive_mean_binary(img):
    return img > box_mean(img, 5)

def _grads(img):
    gx = np.zeros_like(img); gy = np.zeros_like(img)
    gx[:, 1:] = img[:, 1:] - img[:, :-1]
    gy[1:, :] = img[1:, :] - img[:-1, :]
    return gx, gy

def _conv2(img, kern):
    H, W = img.shape
    ph, pw = kern.shape[0] // 2, kern.shape[1] // 2
    p = np.pad(img, ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros_like(img)
    for i in range(kern.shape[0]):
        for j in range(kern.shape[1]):
            out += kern[i, j] * p[i:i + H, j:j + W]
    return out

def canny_binary(img):
    gx, gy = _grads(img)
    mag = np.hypot(gx, gy)
    return mag > mag.mean() + 0.5 * mag.std()

def sobel_binary(img):
    kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
    mag = np.hypot(_conv2(img, kx), _conv2(img, kx.T))
    return mag > mag.mean() + 0.5 * mag.std()

def thermo_binary(img):
    return np.stack([img >= t for t in np.linspace(0.1, 0.9, TM_THERMO_BITS)], -1)

def hog_binary(img):
    gx, gy = _grads(img)
    mag = np.hypot(gx, gy)
    b = ((np.arctan2(gy, gx) % np.pi) / np.pi * TM_HOG_BINS).astype(int) % TM_HOG_BINS
    strong = mag > mag.mean() + 0.5 * mag.std()
    return np.stack([(b == i) & strong for i in range(TM_HOG_BINS)], -1)

def color_binary(img):
    half = 0.15
    return np.stack([(img >= c - half) & (img < c + half)
                     for c in np.linspace(0.15, 0.85, TM_COLOR_WINDOWS)], -1)

METHOD_FUNCS = {"otsu": otsu_binary, "adaptive_mean": adaptive_mean_binary,
                "canny": canny_binary, "sobel": sobel_binary, "thermo": thermo_binary,
                "hog": hog_binary, "color": color_binary}

def slice_features(stack_u8, method):
    """(n_slice, 64, 64) uint8 -> (n_slice, F) uint8 bits at TM_IMG."""
    fn = METHOD_FUNCS[method]
    out = []
    for sl in stack_u8:
        img = _norm01(resize_area(sl.astype(np.float32) / 255.0, TM_IMG))
        out.append(np.asarray(fn(img)).ravel())
    return np.stack(out).astype(np.uint8)

def block_features(cache, slot_k, method):
    """(n_study, n_slice, F) uint8 for one slot and one method."""
    return np.stack([slice_features(cache[i, slot_k], method) for i in range(len(cache))])

## 7. The tabular model — kept, honestly validated, and now the floor

v2's Baseline B (counts of series by plane / fluid sensitivity / fat suppression, plus patient sex)
survives, with two changes: it trains on the §2 report labels rather than v2's weaker ones, and its
CV AUC is computed out-of-fold on the same fold split as the image members so the two are comparable
and can be blended on the same scale. It runs **before** the image stage, not after, because it is
the floor every finding falls back to — including if the kernel runs out of time mid-way through §8.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold

def rank01(v):
    """Everything is combined in rank space: the metric only reads ordering."""
    return pd.Series(np.asarray(v, float)).rank(pct=True).to_numpy()

def safe_auc(y, p):
    y = np.asarray(y).astype(int)
    if len(set(y.tolist())) < 2 or not np.isfinite(p).all() or float(np.std(p)) < 1e-12:
        return np.nan
    return float(roc_auc_score(y, p))

def series_features(series_df):
    uid = series_df["StudyInstanceUID"]
    f = pd.DataFrame({
        "n_series": series_df.groupby(uid).size(),
        "n_fluid":  _as_bool(series_df["Fluid_Sensitive"]).groupby(uid.values).sum(),
        "n_fatsat": _as_bool(series_df["Fat_Suppression"]).groupby(uid.values).sum(),
    })
    f["n_nonfluid"] = f["n_series"] - f["n_fluid"]
    planes = pd.get_dummies(series_df["Anatomical_Plane"], prefix="plane")
    planes["_uid"] = series_df["StudyInstanceUID"].values
    f = f.join(planes.groupby("_uid").sum(), how="outer").fillna(0)
    for p in ["plane_Sagittal", "plane_Coronal", "plane_Axial"]:
        if p not in f:
            f[p] = 0
    f["frac_fluid"] = (f["n_fluid"] / f["n_series"].replace(0, np.nan)).fillna(0)
    f["frac_fatsat"] = (f["n_fatsat"] / f["n_series"].replace(0, np.nan)).fillna(0)
    return f

Xtab_tr = series_features(train_series).reindex(train["StudyInstanceUID"]).fillna(0)
Xtab_te = series_features(test_series).reindex(TEST_UID).fillna(0)
Xtab_te = Xtab_te.reindex(columns=Xtab_tr.columns, fill_value=0)
if "PatientSex" in train.columns:
    Xtab_tr["sex"] = (train.set_index("StudyInstanceUID")["PatientSex"]
                      .map({"Male": 1, "Female": 0}).reindex(Xtab_tr.index).fillna(0.5))
else:
    Xtab_tr["sex"] = 0.5
Xtab_te["sex"] = 0.5

_t_tab = time.time()
tab_oof_auc, tab_test = {}, pd.DataFrame(index=range(len(TEST_UID)))
Xa = Xtab_tr.to_numpy(np.float32)
Xb_ = Xtab_te.to_numpy(np.float32)
for t in LABELS:
    y = Y_REPORT[t].reindex(Xtab_tr.index).fillna(0).to_numpy(int)
    if len(set(y.tolist())) < 2 or y.sum() < 10:
        tab_test[t] = float(PREVALENCE[t]); tab_oof_auc[t] = np.nan
        continue
    oof = np.zeros(len(y))
    for tr, va in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(Xa, y):
        sc = StandardScaler().fit(Xa[tr])
        lr = LogisticRegression(max_iter=1000).fit(sc.transform(Xa[tr]), y[tr])
        hg = HistGradientBoostingClassifier(random_state=SEED, max_iter=200).fit(Xa[tr], y[tr])
        oof[va] = (rank01(lr.predict_proba(sc.transform(Xa[va]))[:, 1])
                   + rank01(hg.predict_proba(Xa[va])[:, 1])) / 2
    tab_oof_auc[t] = safe_auc(y, oof)
    sc = StandardScaler().fit(Xa)
    lr = LogisticRegression(max_iter=1000).fit(sc.transform(Xa), y)
    hg = HistGradientBoostingClassifier(random_state=SEED, max_iter=200).fit(Xa, y)
    tab_test[t] = (rank01(lr.predict_proba(sc.transform(Xb_))[:, 1])
                   + rank01(hg.predict_proba(Xb_)[:, 1])) / 2

STAGE_SECS["tabular model"] = round(time.time() - _t_tab, 1)
print(pd.Series(tab_oof_auc, name="tabular OOF AUC").round(3).to_frame().to_string())
print("macro: %.4f" % np.nanmean(list(tab_oof_auc.values())))

## 8. Members, folds, and the gate

Every **member** is one (slot, method, finding). It is fitted `N_FOLDS` times on grouped folds — the
grouping is by study, so a study's slices are never split across a fold boundary — which yields both
an out-of-fold score for every training study and a test score averaged over the folds.

Then, per finding:

* slice scores are pooled to a study score under **max**, **top-2** and **mean**, and the pooling
  with the best OOF AUC is kept;
* members are combined by **rank mean weighted by `max(0, OOF AUC − 0.5)`** — the same rank-space
  combination v41 uses, for the same reason: the metric only reads ordering;
* if the best member for a finding cannot clear `GATE_AUC`, the image model is **dropped** for that
  finding and the tabular model of §8 takes it.

Members are enrolled in priority order (every finding's first-choice slot before any second choice)
and `submission.csv` is rewritten after each one, so stopping early costs coverage, never validity.

In [ ]:
rng_fold = np.random.default_rng(SEED)

def make_folds(n_study):
    f = np.arange(n_study) % N_FOLDS
    rng_fold.shuffle(f)
    return f

def pool_scores(slice_scores, mode):
    """(n_study, n_slice) -> (n_study,)"""
    if mode == "mean":
        return slice_scores.mean(1)
    if mode == "max":
        return slice_scores.max(1)
    k = min(2, slice_scores.shape[1])
    return np.sort(slice_scores, 1)[:, -k:].mean(1)

POOL_MODES = ["max", "top2", "mean"]

def fit_member(Xtr_sl, ytr, wtr, folds, Xte_sl, seed):
    """Fit one member across the folds. Xtr_sl / Xte_sl: (n_study, n_slice, F) uint8.

    Returns ({pool mode: OOF study scores}, {pool mode: test study scores}).
    """
    n_tr, n_sl, F = Xtr_sl.shape
    flat_tr = Xtr_sl.reshape(-1, F)
    flat_te = Xte_sl.reshape(-1, F)
    oof = np.zeros((n_tr, n_sl), np.float32)
    te_acc = {m: np.zeros(len(Xte_sl), np.float64) for m in POOL_MODES}
    n_used = 0
    for f in range(N_FOLDS):
        tr_studies = np.where(folds != f)[0]
        va_studies = np.where(folds == f)[0]
        if len(tr_studies) < 40 or len(va_studies) == 0:
            continue
        y_tr = ytr[tr_studies]
        if len(set(y_tr.tolist())) < 2:
            continue
        # oversample positives up to POS_FLOOR inside the training half only
        pos = tr_studies[y_tr == 1]
        rate = len(pos) / max(len(tr_studies), 1)
        take = tr_studies
        if 0 < rate < POS_FLOOR and len(pos):
            extra = int(len(tr_studies) * (POS_FLOOR - rate) / max(1 - POS_FLOOR, 1e-6))
            take = np.concatenate([tr_studies, np.resize(pos, extra)])
        rows = (take[:, None] * n_sl + np.arange(n_sl)[None, :]).ravel()
        tm = BatchTsetlinMachine(clauses=TM_CLAUSES, T=TM_T, s=TM_S, epochs=TM_EPOCHS,
                                 batch=TM_BATCH, max_included_literals=TM_MAX_INCLUDED,
                                 weighted=TM_WEIGHTED, seed=seed + f)
        tm.fit(flat_tr[rows],
               np.repeat(ytr[take], n_sl),
               np.repeat(wtr[take], n_sl))
        va_rows = (va_studies[:, None] * n_sl + np.arange(n_sl)[None, :]).ravel()
        oof[va_studies] = tm.decision_function(flat_tr[va_rows]).reshape(len(va_studies), n_sl)
        te_sl = tm.decision_function(flat_te).reshape(len(Xte_sl), n_sl)
        for m in POOL_MODES:
            te_acc[m] += rank01(pool_scores(te_sl, m))
        n_used += 1
        del tm
    if n_used == 0:
        return None, None, 0
    oof_pooled = {m: pool_scores(oof, m) for m in POOL_MODES}
    te_pooled = {m: te_acc[m] / n_used for m in POOL_MODES}
    return oof_pooled, te_pooled, n_used

In [ ]:
IMG_OK = False
member_log = []                      # one row per (finding, slot, method)
img_test = {t: [] for t in LABELS}   # finding -> [(weight, test study scores)]
img_oof  = {t: [] for t in LABELS}   # finding -> [(weight, OOF study scores)] for feedback.txt
skip_log = []                        # members that were never fitted, and why
BLOCKS = dict(planned=0, reached=0, timed_out=False)

if HAVE_IMAGES:
    import pydicom  # noqa: F401  (fail here rather than deep inside the thread pool)

    log("assigning slots")
    _t_slot = time.time()
    assign_tr = slot_assignment(train_series, "train")
    assign_te = slot_assignment(test_series, "test")
    STAGE_SECS["slot assignment"] = round(time.time() - _t_slot, 1)
    log("slots resolved for %d train and %d test studies" % (len(assign_tr), len(assign_te)))

    # Train on the studies the lexicon was most confident about, expert rows first.
    conf_rank = W_REPORT.mean(axis=1)
    if len(GOLD):
        conf_rank.loc[GOLD.index] = 2.0
    order = conf_rank.sort_values(ascending=False).index
    tr_uids = [u for u in order if u in assign_tr][:N_TRAIN]

    tr_uids, C_tr, M_tr = build_cache(tr_uids, assign_tr, "train", "train")
    te_uids, C_te, M_te = build_cache(TEST_UID, assign_te, "test", "test")
    # build_cache drops test studies with no usable series, so keep the map back
    # into submission row order -- the two are not the same length.
    _te_row = {u: i for i, u in enumerate(TEST_UID)}
    TE_POS = np.array([_te_row[u] for u in te_uids], int)
    IMG_OK = len(tr_uids) >= MIN_TRAIN and len(te_uids) > 0
    if not IMG_OK:
        log("not enough usable images (%d train / %d test); the image model is skipped"
            % (len(tr_uids), len(te_uids)))
else:
    log("no DICOMs mounted (run on Kaggle for the image model); CSV models only")

In [ ]:
def combine_predictions():
    """Per finding: gated image members + the tabular floor, rank-mean weighted by OOF AUC."""
    pred = pd.DataFrame({"StudyInstanceUID": TEST_UID})
    chosen = {}
    for t in LABELS:
        members = [(w, v) for w, v in img_test.get(t, []) if w > 0]
        best_img = max([m["oof_auc"] for m in member_log
                        if m["finding"] == t and np.isfinite(m["oof_auc"])], default=np.nan)
        w_tab = 0.0 if not np.isfinite(tab_oof_auc.get(t, np.nan)) else max(0.0, tab_oof_auc[t] - 0.5)
        use_img = bool(members) and np.isfinite(best_img) and best_img >= GATE_AUC

        acc = np.zeros(len(TEST_UID)); tot = 0.0
        if use_img:
            for w, v in members:
                # a study missing this slot gets the member's median rank, i.e. no vote
                filled = np.where(np.isfinite(v), v, np.nanmedian(v) if np.isfinite(v).any() else 0.5)
                acc += w * filled; tot += w
        if w_tab > 0:
            acc += w_tab * rank01(np.asarray(tab_test[t], float)); tot += w_tab
        if tot <= 0:
            pred[t] = float(PREVALENCE[t]); chosen[t] = "prevalence"
        else:
            v = acc / tot
            lo, hi = float(v.min()), float(v.max())
            pred[t] = 0.5 if hi <= lo else (v - lo) / (hi - lo)
            chosen[t] = ("image+tabular" if use_img and w_tab > 0 else
                         "image" if use_img else "tabular")
    combine_predictions.source = chosen
    return pred

def bank_submission(reason=""):
    """Rewrite submission.csv from whatever is combined so far."""
    pred = combine_predictions()
    sub = pred[sample_sub.columns].copy()
    assert list(sub.columns) == list(sample_sub.columns)
    assert len(sub) == len(test) and sub["StudyInstanceUID"].is_unique
    sub.to_csv("submission.csv", index=False)
    if reason:
        log("banked submission.csv (%s)" % reason)
    return sub

### 8b. `feedback.txt` — the measurements v5 is planned from

Everything above produces numbers that only exist inside the kernel. This writes them to
**`feedback.txt`** (with a machine-readable `feedback.json` beside it) in the notebook's output
directory, so they come back with the run and the next iteration can be argued from measurements
rather than guesses. It is rewritten after every block, so a kernel that runs out of time still
leaves a complete report of everything finished by then.

The report answers, per finding: *did the image model learn anything, off which sequence, with which
feature map, pooled how, and what is stopping it* — plus the two questions that decide where v5's
effort goes:

* **Is it labels or is it pixels?** Every member's OOF AUC is reported twice, once over all studies
  and once over only the studies the lexicon was confident about. A large gap means the ceiling is
  label noise and v5 should buy better labels; no gap means the representation is the limit.
* **Is the ensemble earning its runtime?** The blended OOF AUC is compared against the best single
  member, per finding, and mean AUC is broken down by feature map and by slot — so members that
  never pay for themselves can be dropped instead of guessed at.

In [ ]:
def _fmt_table(df, floatfmt="%.3f"):
    if df is None or not len(df):
        return "    (nothing to report)\n"
    return "".join("    " + l + "\n" for l in
                   df.to_string(index=False, float_format=lambda v: floatfmt % v).splitlines())

def blend_oof(t):
    """Blend a finding's image members exactly as combine_predictions does, but OOF."""
    members = [(w, v) for w, v in img_oof.get(t, []) if w > 0]
    if not members:
        return None
    acc, tot = np.zeros(len(members[0][1])), 0.0
    for w, v in members:
        acc += w * np.where(np.isfinite(v), v, np.nanmedian(v) if np.isfinite(v).any() else 0.5)
        tot += w
    return acc / tot if tot > 0 else None

def write_feedback(path="feedback.txt"):
    L, J = [], {}
    def head(s):
        L.append("")
        L.append("=" * 78)
        L.append(s)
        L.append("=" * 78)

    n_mem = len(member_log)
    L.append("RSNA knee -- CPU Tsetlin Machine v4 -- feedback report")
    L.append("written %.1f min into the run; %d image member(s) enrolled so far"
             % ((time.time() - T0) / 60, n_mem))
    L.append("read this top to bottom; section 8 is the shortlist for v5.")

    # ---- 1. configuration ------------------------------------------------
    cfg = dict(CACHE_IMG=CACHE_IMG, TM_IMG=TM_IMG, CROP_MM=CROP_MM, SLICE_BAND=SLICE_BAND,
               N_SLICE=N_SLICE, TM_CLAUSES=TM_CLAUSES, TM_T=TM_T, TM_S=TM_S,
               TM_EPOCHS=TM_EPOCHS, TM_BATCH=TM_BATCH, TM_MAX_INCLUDED=TM_MAX_INCLUDED,
               TM_WEIGHTED=TM_WEIGHTED, TM_METHODS=TM_METHODS, N_TRAIN=N_TRAIN,
               N_FOLDS=N_FOLDS, POS_FLOOR=POS_FLOOR, GATE_AUC=GATE_AUC,
               SLOTS_PER_TARGET=SLOTS_PER_TARGET, TIME_BUDGET_H=round(TIME_BUDGET / 3600, 2),
               n_cpu=os.cpu_count())
    head("1. THE CONFIGURATION THAT PRODUCED THESE NUMBERS")
    for k, v in cfg.items():
        L.append("    %-18s %s" % (k, v))
    J["config"] = {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}

    # ---- 2. timing -------------------------------------------------------
    head("2. WHERE THE TIME WENT")
    for k, v in STAGE_SECS.items():
        L.append("    %-18s %8.1f s  (%.1f min)" % (k, v, v / 60))
    L.append("    %-18s %8.1f s  (%.1f min)" % ("elapsed total", time.time() - T0,
                                                (time.time() - T0) / 60))
    L.append("    time budget left: %.1f min" % (time_left() / 60))
    if BLOCKS["planned"]:
        L.append("    blocks reached: %d of %d%s"
                 % (BLOCKS["reached"], BLOCKS["planned"],
                    "  *** the rest were dropped for time ***" if BLOCKS["timed_out"]
                    else "  (any shortfall is a slot no study had -- see section 5)"))
    if DECODE_STATS:
        L.append("")
        L.append("    DICOM decoding:")
        for tag, d in DECODE_STATS.items():
            L.append("      %-6s %d studies, %d/%d slot-series filled (%.0f%%), %.0f s%s"
                     % (tag, d["studies"], d["filled"], d["jobs"],
                        100.0 * d["filled"] / max(d["jobs"], 1), d["secs"],
                        "  *** TRUNCATED BY BUDGET ***" if d["truncated"] else ""))
    J["stage_secs"] = dict(STAGE_SECS)
    J["decode"] = DECODE_STATS

    # ---- 3. label quality ------------------------------------------------
    head("3. LABEL QUALITY -- the report lexicon is the training set")
    lab_tab = pd.DataFrame({
        "finding": LABELS,
        "derived_pos_rate": [round(float(Y_REPORT[t].mean()), 3) for t in LABELS],
        "mean_conf": [round(float(W_REPORT[t].mean()), 3) for t in LABELS],
        "silence_rate": [round(float(SILENT[t].mean()), 3) for t in LABELS],
        "clauses_pos": [int((LAB[t + "__npos"] > 0).sum()) for t in LABELS],
        "clauses_neg": [int((LAB[t + "__nneg"] > 0).sum()) for t in LABELS],
    })
    if len(AGREE):
        lab_tab = lab_tab.merge(AGREE[["finding", "agreement AUC", "npos", "nneg"]],
                                on="finding", how="left")
    L.append(_fmt_table(lab_tab))
    L.append("    'silence_rate' = share of studies where no rule fired at all: pure guesswork")
    L.append("    for that finding. 'agreement AUC' is against the %d annotated studies only."
             % len(GOLD))
    L.append("")
    L.append("    Silence by report language (%, one column per finding):")
    L.append(_fmt_table(SILENCE_BY_LANG.reset_index().rename(columns={"index": "lang"}), "%.0f"))
    L.append("    A finding silent in one language and not another is a lexicon gap, and closing")
    L.append("    it adds training rows for free. A finding silent everywhere is a reporting")
    L.append("    convention, and no lexicon work will fix it.")
    J["labels"] = lab_tab.to_dict("records")
    J["silence_by_language"] = SILENCE_BY_LANG.reset_index().to_dict("records")

    # ---- 4. image coverage ----------------------------------------------
    head("4. IMAGE COVERAGE -- do the slots match what the studies actually contain?")
    inv = []
    for tag, sdf in (("train", train_series), ("test", test_series)):
        g = sdf.copy()
        g["plane"] = g["Anatomical_Plane"].astype(str).str.strip().str.title()
        g["fluid"] = _as_bool(g["Fluid_Sensitive"]) if "Fluid_Sensitive" in g else False
        g["fatsat"] = _as_bool(g["Fat_Suppression"]) if "Fat_Suppression" in g else False
        n_st = g["StudyInstanceUID"].nunique()
        for (p, fl, fs), gg in g.groupby(["plane", "fluid", "fatsat"]):
            inv.append(dict(split=tag, plane=p, fluid=bool(fl), fatsat=bool(fs),
                            n_series=len(gg),
                            pct_studies=round(100.0 * gg["StudyInstanceUID"].nunique() / max(n_st, 1), 1)))
    L.append("    Series inventory (what exists, whether or not a slot asks for it):")
    L.append(_fmt_table(pd.DataFrame(inv), "%.1f"))
    J["series_inventory"] = inv
    if "M_tr" in globals():
        cov = pd.DataFrame({
            "slot": SLOT_NAMES,
            "plane": [SLOT_PLANE[s] for s in SLOT_NAMES],
            "pct_train_studies": (100.0 * M_tr.mean(0)).round(1),
            "pct_test_studies": (100.0 * M_te.mean(0)).round(1)})
        L.append("    Slot fill rate after fallback:")
        L.append(_fmt_table(cov, "%.1f"))
        J["slot_coverage"] = cov.to_dict("records")

    # ---- 5. member results ----------------------------------------------
    head("5. EVERY IMAGE MEMBER (finding x slot x feature map)")
    if n_mem:
        md_ = pd.DataFrame(member_log).drop(columns=["auc_by_pool"])
        L.append(_fmt_table(md_.sort_values("oof_auc", ascending=False)))
        L.append("    oof_auc          out-of-fold AUC against the report labels, grouped by study")
        L.append("    oof_auc_highconf the same, over only the studies the lexicon was sure about")
        L.append("    weight           max(0, oof_auc - 0.5): its vote in the rank-mean blend")
        J["members"] = [{k: v for k, v in m.items() if k != "auc_by_pool"} for m in member_log]
        J["members_auc_by_pool"] = [dict(finding=m["finding"], slot=m["slot"],
                                         method=m["method"], **m["auc_by_pool"])
                                    for m in member_log]
    else:
        L.append("    No image members were fitted (no DICOMs mounted, or the budget ran out).")

    if skip_log:
        L.append("")
        L.append("    Members that were never fitted:")
        sk = pd.DataFrame(skip_log)
        L.append(_fmt_table(sk.groupby("reason").size().reset_index(name="n_members")))
        L.append("    by finding: " + ", ".join(
            "%s x%d" % (k, v) for k, v in sk["finding"].value_counts().items()))
        J["skipped"] = skip_log

    # ---- 6. per-finding summary -----------------------------------------
    head("6. PER FINDING -- what is serving it, and what is stopping it")
    rows = []
    for t in LABELS:
        ms = [m for m in member_log if m["finding"] == t and np.isfinite(m["oof_auc"])]
        best = max(ms, key=lambda m: m["oof_auc"]) if ms else None
        bl = blend_oof(t)
        y_oof = (Y_REPORT.reindex(tr_uids)[t].to_numpy(int)
                 if ("tr_uids" in globals() and bl is not None) else None)
        blend_auc = safe_auc(y_oof, bl) if bl is not None and y_oof is not None else np.nan
        rows.append(dict(
            finding=t,
            best_member_auc=round(best["oof_auc"], 3) if best else np.nan,
            blend_auc=round(blend_auc, 3) if np.isfinite(blend_auc) else np.nan,
            blend_gain=(round(blend_auc - best["oof_auc"], 3)
                        if best and np.isfinite(blend_auc) else np.nan),
            highconf_auc=round(best["oof_auc_highconf"], 3)
                         if best and np.isfinite(best.get("oof_auc_highconf", np.nan)) else np.nan,
            best_slot=best["slot"] if best else "-",
            best_map=best["method"] if best else "-",
            pool=best["pool"] if best else "-",
            n_pos=best["n_pos"] if best else int(Y_REPORT[t].sum()),
            tabular_auc=round(tab_oof_auc.get(t, np.nan), 3),
            serving=(combine_predictions.source.get(t, "-")
                     if hasattr(combine_predictions, "source") else "-")))
    summary = pd.DataFrame(rows)
    L.append(_fmt_table(summary))
    img_ok_aucs = summary["best_member_auc"].dropna()
    if len(img_ok_aucs):
        L.append("    macro over findings with an image member: %.4f" % img_ok_aucs.mean())
    L.append("    macro tabular: %.4f" % np.nanmean(list(tab_oof_auc.values())))
    J["per_finding"] = summary.to_dict("records")

    # ---- 7. knob diagnostics --------------------------------------------
    head("7. WHICH KNOBS PAID -- evidence for the next set of settings")
    if n_mem:
        mdf = pd.DataFrame(member_log)
        by_map = (mdf.groupby("method")["oof_auc"]
                  .agg(["mean", "max", "count"]).reset_index()
                  .rename(columns={"method": "feature_map", "mean": "mean_auc", "max": "best_auc",
                                   "count": "n_members"}))
        by_map["mean_secs"] = mdf.groupby("method")["secs"].mean().values.round(1)
        L.append("    Feature map (a map whose mean AUC is at 0.50 is paying nothing):")
        L.append(_fmt_table(by_map))
        by_slot = (mdf.groupby("slot")["oof_auc"].agg(["mean", "max", "count"]).reset_index()
                   .rename(columns={"mean": "mean_auc", "max": "best_auc", "count": "n_members"}))
        L.append("    Slot:")
        L.append(_fmt_table(by_slot))
        by_rank = (mdf.groupby("slot_rank")["oof_auc"].agg(["mean", "count"]).reset_index()
                   .rename(columns={"mean": "mean_auc", "count": "n_members"}))
        L.append("    Slot prior rank (0 = the slot SLOT_PRIOR names first):")
        L.append(_fmt_table(by_rank))
        L.append("    Pooling chosen (max winning means the evidence is focal: more slices help):")
        L.append(_fmt_table(mdf["pool"].value_counts().rename_axis("pool")
                            .reset_index(name="n_members")))
        gap = (mdf["oof_auc_highconf"] - mdf["oof_auc"]).dropna()
        if len(gap):
            L.append("    Label-noise probe: high-confidence AUC minus all-studies AUC,")
            L.append("      mean %+0.3f over %d members (positive => labels are the ceiling)."
                     % (gap.mean(), len(gap)))
            J["label_noise_gap"] = round(float(gap.mean()), 4)
        per_member = mdf["secs"].mean()
        L.append("")
        L.append("    Cost model: %d members at %.0f s each. Rough multipliers for v5 --"
                 % (n_mem, per_member))
        L.append("      N_SLICE %d -> %d   ~x%.1f      TM_IMG %d -> 48  ~x%.1f"
                 % (N_SLICE, N_SLICE * 2, 2.0, TM_IMG, (48.0 / TM_IMG) ** 2))
        L.append("      TM_EPOCHS %d -> %d  ~x2.0      SLOTS_PER_TARGET %d -> %d  ~x%.1f"
                 % (TM_EPOCHS, TM_EPOCHS * 2, SLOTS_PER_TARGET, SLOTS_PER_TARGET + 1,
                    (SLOTS_PER_TARGET + 1) / max(SLOTS_PER_TARGET, 1)))
        J["by_map"] = by_map.to_dict("records")
        J["by_slot"] = by_slot.to_dict("records")

    # ---- 8. shortlist ----------------------------------------------------
    head("8. SHORTLIST FOR v5 (generated from the numbers above)")
    tips = []
    if n_mem:
        mdf = pd.DataFrame(member_log)
        strong = summary[summary["best_member_auc"] >= 0.60]["finding"].tolist()
        weak = summary[(summary["best_member_auc"].notna())
                       & (summary["best_member_auc"] < GATE_AUC)]["finding"].tolist()
        if strong:
            tips.append("CAPACITY: %s clear 0.60. Spend the next budget here: TM_IMG 48, "
                        "more clauses, a third slot." % ", ".join(strong))
        if weak:
            starved = [t for t in weak
                       if float(summary.loc[summary.finding == t, "n_pos"].iloc[0]) < 100]
            if starved:
                tips.append("STARVED: %s failed the gate with under 100 positives. That is a label "
                            "problem, not a model problem -- widen the lexicon or add a second "
                            "label source before touching the machine." % ", ".join(starved))
            rest = [t for t in weak if t not in starved]
            if rest:
                tips.append("REPRESENTATION: %s failed the gate with enough positives, so the "
                            "signal is not surviving binarisation at %d px. Localise the crop "
                            "(intercondylar region) before raising resolution."
                            % (", ".join(rest), TM_IMG))
        # is the prior naming the right slot first?
        for t in LABELS:
            ms = mdf[(mdf.finding == t) & mdf.oof_auc.notna()]
            if ms["slot_rank"].nunique() > 1:
                best_row = ms.loc[ms.oof_auc.idxmax()]
                if best_row["slot_rank"] != 0:
                    tips.append("SLOT_PRIOR: %s scores best on %s (rank %d, AUC %.3f), not on the "
                                "slot named first. Reorder SLOT_PRIOR['%s']."
                                % (t, best_row["slot"], best_row["slot_rank"],
                                   best_row["oof_auc"], t))
        dead = [r["feature_map"] for _, r in by_map.iterrows() if r["mean_auc"] < 0.52]
        if dead:
            tips.append("FEATURE MAPS: %s average below 0.52 -- drop them from TM_METHODS and try "
                        "the unused ones (%s)." % (", ".join(dead),
                        ", ".join(m for m in METHOD_FUNCS if m not in TM_METHODS)))
        if (mdf["pool"] == "max").mean() > 0.5:
            tips.append("SLICES: max pooling won for most members, so the evidence sits on a few "
                        "slices. Raising N_SLICE from %d to 7 is the cheapest gain available -- "
                        "the cache is already %d px and no re-decode is needed."
                        % (N_SLICE, CACHE_IMG))
        gap = (mdf["oof_auc_highconf"] - mdf["oof_auc"]).dropna()
        if len(gap) and gap.mean() > 0.03:
            tips.append("LABELS ARE THE CEILING: members score %+0.3f higher on studies the "
                        "lexicon was sure about. A second report reading (v41 blends three) "
                        "would lift every member at once." % gap.mean())
        blend_gain = summary["blend_gain"].dropna()
        if len(blend_gain) and blend_gain.mean() < 0.005:
            tips.append("ENSEMBLE: blending adds only %+0.3f over the best single member. Fit "
                        "fewer, better members rather than more of them." % blend_gain.mean())
    hi_sil = [t for t in LABELS if float(SILENT[t].mean()) > 0.5]
    if hi_sil:
        tips.append("LEXICON: no rule fires at all for %s on over half the corpus. Check section 3's "
                    "language table -- if the silence is concentrated in one language it is a "
                    "vocabulary gap worth closing." % ", ".join(hi_sil))
    if BLOCKS["timed_out"]:
        tips.append("BUDGET: only %d of %d blocks were fitted before time ran out. Either raise "
                    "RSNA_TIME_BUDGET, or cut the cost per member (fewer TM_EPOCHS, fewer "
                    "TM_METHODS) so the whole grid fits -- an unfitted finding falls back to "
                    "the tabular model." % (BLOCKS["reached"], BLOCKS["planned"]))
    if any(d["truncated"] for d in DECODE_STATS.values()):
        tips.append("BUDGET: DICOM decoding was cut short. Raise RSNA_TIME_BUDGET or lower N_TRAIN; "
                    "the studies that were dropped had no vote.")
    if not tips:
        tips.append("Nothing stands out automatically -- read section 6 by hand.")
    for i, tip in enumerate(tips, 1):
        L.append("    %2d. %s" % (i, tip))
    J["shortlist"] = tips

    L.append("")
    L.append("end of report")
    Path(path).write_text("\n".join(L), encoding="utf-8")
    try:
        Path(path).with_suffix(".json").write_text(
            json.dumps(J, indent=1, default=str), encoding="utf-8")
    except (TypeError, ValueError):
        pass
    return L

In [ ]:
if IMG_OK:
    _t_mem = time.time()
    y_all = Y_REPORT.reindex(tr_uids)
    w_all = W_REPORT.reindex(tr_uids).fillna(0.0)
    folds = make_folds(len(tr_uids))
    slot_of = {n: k for k, n in enumerate(SLOT_NAMES)}

    # (slot rank, slot, method) blocks: every finding's first-choice slot comes first
    blocks = []
    for rank in range(SLOTS_PER_TARGET):
        for name in SLOT_NAMES:
            tgts = [t for t in LABELS
                    if len(SLOT_PRIOR[t]) > rank and SLOT_PRIOR[t][rank] == name]
            if tgts:
                blocks += [(rank, name, method, tgts) for method in TM_METHODS]

    BLOCKS["planned"] = len(blocks)
    for n_block, (rank, name, method, tgts) in enumerate(blocks):
        BLOCKS["reached"] = n_block + 1
        if time_left() < 300:
            BLOCKS["timed_out"] = True
            BLOCKS["reached"] = n_block
            log("time budget spent after %d of %d blocks; the rest are not enrolled"
                % (n_block, len(blocks)))
            skip_log += [dict(finding=t, slot=n_[1], method=n_[2], reason="time budget")
                         for n_ in blocks[n_block:] for t in n_[3]]
            break
        k = slot_of[name]
        have_tr = M_tr[:, k] > 0.5
        have_te = M_te[:, k] > 0.5
        if have_tr.sum() < MIN_TRAIN or have_te.sum() < 1:
            log("slot %s: only %d train / %d test studies have it; skipped"
                % (name, int(have_tr.sum()), int(have_te.sum())))
            skip_log += [dict(finding=t, slot=name, method=method,
                              reason="slot present on only %d train / %d test studies"
                                     % (int(have_tr.sum()), int(have_te.sum()))) for t in tgts]
            continue
        t_blk = time.time()
        Xtr = block_features(C_tr[have_tr], k, method)
        Xte = block_features(C_te[have_te], k, method)
        log("block slot=%s method=%s F=%d (%d train / %d test studies, %.0fs to binarise)"
            % (name, method, Xtr.shape[2], len(Xtr), len(Xte), time.time() - t_blk))

        for t in tgts:
            if time_left() < 240:
                break
            y = y_all[t].values[have_tr].astype(int)
            w = np.clip(w_all[t].values[have_tr], 0.05, 1.0)
            if len(set(y.tolist())) < 2 or y.sum() < 15 or (1 - y).sum() < 15:
                log("  %-17s too few positives (%d) on this slot; skipped" % (t, int(y.sum())))
                skip_log.append(dict(finding=t, slot=name, method=method,
                                     reason="only %d positive / %d negative studies"
                                            % (int(y.sum()), int((1 - y).sum()))))
                continue
            t_m = time.time()
            oof_p, te_p, n_folds_used = fit_member(Xtr, y, w, folds[have_tr], Xte,
                                                   seed=SEED + 101 * LABELS.index(t))
            if oof_p is None:
                skip_log.append(dict(finding=t, slot=name, method=method,
                                     reason="no fold had both classes"))
                continue
            aucs = {m: safe_auc(y, oof_p[m]) for m in POOL_MODES}
            best = max(POOL_MODES, key=lambda m: (aucs[m] if np.isfinite(aucs[m]) else -1))
            auc = aucs[best]
            wgt = 0.0 if not np.isfinite(auc) else max(0.0, auc - 0.5)
            # Same AUC restricted to the studies the lexicon was confident about. If this
            # is much higher than the AUC over all studies, label noise is the binding
            # constraint and v5 should buy labels, not clauses.
            hi_conf = w >= max(0.5, float(np.quantile(w, 0.6)))
            auc_hi = (safe_auc(y[hi_conf], oof_p[best][hi_conf])
                      if hi_conf.sum() >= 40 else np.nan)
            member_log.append(dict(finding=t, slot=name, method=method, slot_rank=rank,
                                   pool=best, oof_auc=auc, oof_auc_highconf=auc_hi,
                                   weight=wgt, n_pos=int(y.sum()), n_studies=int(len(y)),
                                   n_features=int(Xtr.shape[2]), folds=n_folds_used,
                                   auc_by_pool={m: aucs[m] for m in POOL_MODES},
                                   secs=round(time.time() - t_m, 1)))
            log("  %-17s slot=%-6s %-13s pool=%-5s OOF AUC %.3f  (%.0fs)"
                % (t, name, method, best, auc if np.isfinite(auc) else float("nan"),
                   time.time() - t_m))
            if wgt > 0:
                e = np.full(len(TEST_UID), np.nan)
                e[TE_POS[have_te]] = rank01(te_p[best])
                img_test[t].append((wgt, e))
                o = np.full(len(tr_uids), np.nan)
                o[have_tr] = rank01(oof_p[best])
                img_oof[t].append((wgt, o))
        del Xtr, Xte
        gc.collect()
        bank_submission()
        write_feedback()          # cheap, and survives a kernel that runs out of time
    STAGE_SECS["image members"] = round(time.time() - _t_mem, 1)
    if member_log:
        print(pd.DataFrame(member_log).sort_values("oof_auc", ascending=False).round(3).to_string(index=False))

## 9. Submission and the feedback report

`combine_predictions` already ran after every member above; this writes the final `submission.csv`
and the complete `feedback.txt` / `feedback.json` diagnostics described in §8b.

In [ ]:
submission = bank_submission("final")
print(pd.Series(combine_predictions.source, name="source per finding").to_frame().to_string())
print()
print(submission.head())

chk = pd.read_csv("submission.csv", dtype={"StudyInstanceUID": str})
print("\nround-trip:", chk.shape,
      "| in [0,1]:", bool(chk[LABELS].min().min() >= 0 and chk[LABELS].max().max() <= 1),
      "| constant columns:", [c for c in LABELS if chk[c].nunique() <= 1])

In [ ]:
# The full diagnostics: written to disk, and echoed here so they are in the log too.
REPORT = write_feedback("feedback.txt")
print("\n".join(REPORT))
print("\n" + "-" * 78)
print("written: feedback.txt, feedback.json, submission.csv")
print("On Kaggle these land in /kaggle/working and come back with the run -- open the")
print("notebook's Output tab, download feedback.txt, and start the next iteration from it.")
print("OOF AUC throughout is measured against the report-derived labels, which are themselves")
print("imperfect: read it as a ranking of findings and as a gate, not as a leaderboard estimate.")
print("total runtime: %.1f min" % ((time.time() - T0) / 60))

## 10. Where v5 should go

**Start from `feedback.txt`.** Section 8 of that file is a shortlist generated from this run's own
numbers, and sections 5–7 are the evidence behind it. The list below is the standing menu; the file
tells you which items this run actually earned.

1. **Read the OOF table before touching anything.** Findings above ~0.60 are learnable at this
   resolution and deserve more capacity (`TM_CLAUSES`, `TM_EPOCHS`, `TM_IMG` 48, a third slot).
   Findings stuck at 0.50 will not be rescued by more clauses — they need either localisation or a
   different representation.
2. **More slices, and a slice-position feature.** `N_SLICE` 3 → 7 with max pooling is the cheapest
   real gain for focal findings; the cache is already 64 px, so it costs decode time only. Adding
   the normalised slice index as extra bits lets clauses learn *where* in the stack to look.
3. **Localise before binarising.** The joint centre is not the image centre. A cheap centre-of-mass
   or intensity-projection crop to the intercondylar region would raise the effective resolution far
   more than doubling `TM_IMG` does, at no runtime cost.
4. **Per-compartment crops for OA.** Medial / lateral OA differ only in *which half* of a coronal
   slice is affected; feeding each machine its own half-image makes the two findings separable
   instead of near-duplicate.
5. **More label sources.** v41's pipeline blends three independently-authored report readings and
   trusts them where they agree. A second miner, or an LLM pass over the reports run offline and
   attached as a dataset, would raise the ceiling on *every* image member at once — the labels are
   still the binding constraint here, not the machine.
6. **If a GPU is ever allowed**, note where the remaining gap actually is: v41 gets its 0.91 from a
   pretrained ViT with cross-series attention and a 25-model ensemble. The CPU/efficiency track is
   where this notebook competes.